# La publicité fait-elle vendre ?

Ce test proposé par Numberly consite en 2 étapes :
1) Comprendre le marché ;
2) Comprendre l’impact de la publicité sur les ventes.

Commençons par faire un point sur les données à disposition.

**Convention de nommage des tables :**
| Niveau | Noms | Contenu | Sert à |
|---|---|---|---|
| A | `retailer`, `programmatic_pub`, `tv_pub`, `socio_demo`, `mapping` | Brut, jamais modifié | Audit uniquement |
| B | `retailer_c`, `prog_c`, `tv_c`, `socio_c`, `mapping_c` | Doublons retirés, types convertis, "unknown" remplacés par NaN | Totaux : CA, budget média, funnel, saisonnalité |
| C | `retailer_id`, `prog_id`, `tv_id`, `socio_id`, `mapping_id` | Niveau B sans les lignes dont l'identifiant est manquant | Tout ce qui passe par un client : jointures, attribution, panier moyen par acheteur |

## 0) Chargement des données

**Bibliothèques nécessaire :**

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

**Chargement des données :**

In [ ]:
retailer = pd.read_csv("retailer.csv")
programmatic_pub = pd.read_csv("programmatic_publisher.csv")
tv_pub = pd.read_csv("tv_publisher.csv")
socio_demo = pd.read_csv("socio_demo.csv")
mapping = pd.read_csv("mapping_transac_publisher_tv.csv")

datasets = {
    "retailer": retailer,
    "programmatic_pub": programmatic_pub,
    "tv_pub": tv_pub,
    "socio_demo": socio_demo,
    "mapping": mapping,
}

print("Données chargées :")
for name, df in datasets.items():
    print(f"  {name:<18}: {df.shape}")

## 1) Audit des données

Toutes les vérifications de cette section portent sur les tables brutes (niveau A), sans aucune modification.

### Types des variables :

In [ ]:
for name, df in datasets.items():
    print(f"\n{'=' * 50}\n{name}\n{'=' * 50}")
    df.info()

Point de vigilance : 
- les types de certaines variables ne sont pas toujours corrects (ex : les variables de temps doivent être converties).

### Statistiques de base :

In [ ]:
for name, df in datasets.items():
    print(f"\n{'=' * 50}\n{name}\n{'=' * 50}")
    print(df.describe(include="all").round(2))

Points de vigilance : 
- les identifiants (`customer_id`, `dsp_id`, `device_id`) contiennent tous la valeur "unknown", qui équivaut à une valeur manquante;
- les variables `sales` et `quantity` de la table retailer ont comme valeur minimale 0.

### Valeurs manquantes :

In [ ]:
for name, df in datasets.items():
    missing = pd.DataFrame({
        "Valeurs manquantes": df.isna().sum(),
        "Pourcentage (%)": (df.isna().mean() * 100).round(0).astype(int)
    })
    print(f"\n{'=' * 50}\n{name}\n{'=' * 50}")
    print(missing)

#### Zoom sur la table `retailer`

Chaque ligne est un événement enregistré sur le site du distributeur pour un client et un produit donné.  
Regardons les types d'événements pour comprendre les valeurs manquantes.

In [ ]:
retailer.sample(5)

In [ ]:
pd.DataFrame({
    "Lignes": retailer["event_name"].value_counts(),
    "Part (%)": (retailer["event_name"].value_counts(normalize=True) * 100).round(0).astype(int)
})


Trois étapes successives existent :

| `event_name`      | Lignes    | Part   | Signification                        |
|-------------------|-----------|--------|--------------------------------------|
| Product Page View | 6 376 627 | 65   % | Le client consulte une fiche produit |
| Add to cart       | 2 050 111 | 21   % | Il ajoute le produit au panier       |
| Order             | 1 439 311 | 15   % | Il achète                            |

- Variables `sales` et `quantity` :

On ne peut parler de montant et de quantité que s'il y a eu une vente : les valeurs manquantes de `sales` et `quantity` semblent être structurelles. Vérifions-le.

In [ ]:
missing_sales_qt = retailer[retailer[["sales", "quantity"]].isna().any(axis=1)]
print("Événements associés aux valeurs manquantes de sales / quantity :")
print(missing_sales_qt["event_name"].value_counts(dropna=False))

Seuls les événements `Product Page View` et `Add to cart` sont concernés, et aucun `Order` : les manquants sont bien structurels et ne posent pas de problème.  

**Décision :** aucune imputation ; les KPI de vente se calculent sur event_name == "Order".

- Variables `brand` et `product_name` :

In [ ]:
missing_brand = retailer[retailer["brand"].isna()]
missing_product = retailer[retailer["product_name"].isna()]
print("Exactement les mêmes lignes :", missing_brand.equals(missing_product))


Ce sont les mêmes lignes qui sont concernées par les valeurs manquantes de ces deux variables. Quand l’un manque, l’autre manque aussi.

In [ ]:
events = pd.DataFrame({
    "Valeurs manquantes": missing_brand["event_name"].value_counts(dropna=False)
})
# Nombre total de lignes par événement dans le dataset complet
events["Total événement"] = retailer["event_name"].value_counts(dropna=False)
# Pourcentage de valeurs manquantes par événement
events["Pourcentage (%)"] = (
    events["Valeurs manquantes"] / events["Total événement"] * 100
).round(1)

print(events)


Les manquants se trouvent dans les trois types d’événements et représentent qu'une faible part de celles-ci.

___________________ 

|>>>

In [ ]:
# Poids dans les ventes
orders = retailer[retailer['event_name'] == 'Order']

poids = orders.groupby(orders['brand'].isna())['sales'].agg(['count', 'sum','mean']).round().astype(int)
poids["part du CA (%)"] = (poids["sum"] / poids["sum"].sum() * 100).round(2)
print(poids)

In [ ]:
# Répartition dans le temps et chez les clients
mois_total = retailer["timestamp_utc"].str[:7].value_counts().sort_index()
mois_manq = missing_brand["timestamp_utc"].str[:7].value_counts().sort_index()
print(pd.DataFrame({"manquants": mois_manq, "part (%)": (mois_manq / mois_total * 100).round(2)}))

print("\nClients concernés :", missing_brand["customer_id"].nunique(),
      "sur", retailer["customer_id"].nunique(), "clients distincts")

*Poids dans les ventes. Les 8 776 commandes sans marque pèsent 699 706 € sur un total de 62,43 M€, soit 1,1 % du CA. 
Un point à noter. Ces commandes ont un montant moyen d’environ 80 € par ligne, contre environ 43 € pour les autres. Elles sont donc moins nombreuses mais plus chères que la moyenne. Ça n’invalide rien, mais ça empêche de dire qu’elles sont « comparables aux autres ».
Dans le temps. Les manquants sont présents tous les mois (de janvier à juin 2024), sans pic isolé. Ça exclut un bug ponctuel sur un mois. Ils augmentent de 33 k à 59 k par mois, mais ça peut simplement suivre la hausse de l’activité globale du site.
Chez les clients. 134 172 clients distincts sont concernés pour 277 240 lignes, soit environ 2 lignes par client. Ce n’est donc pas quelques comptes qui posent problème, c’est diffus.*

*à faire: 2. Compare aussi à la population totale : retailer['customer_id'].nunique(), pour dire quelle part des clients est concernée.*

*Marque et produit manquants (277 240 lignes, 2,8 %). Les deux colonnes sont manquantes simultanément, ce qui suggère une absence d’information produit à la source. Ces lignes se répartissent sur tous les types d’événements (3,8 % des vues de page, 1,2 % des ajouts au panier, 0,6 % des commandes), sur l’ensemble de la période (janvier à juin 2024) et sur 134 172 clients différents : il n’y a ni pic temporel ni concentration sur quelques comptes, ce qui écarte un incident ponctuel. Côté ventes, les 8 776 commandes concernées représentent 0,70 M€, soit 1,1 % du CA total. Leur montant moyen par ligne (~80 €) est supérieur à celui des autres commandes (~43 €), ce qui peut indiquer un type de produit particulier non catalogué.*

**Décision :** aucune imputation. Ces lignes sont exclues des analyses par marque et par produit, mais conservées pour les analyses au niveau client et pour le CA total.

|<<
____________________________

#### Zoom sur la table `mapping`

La variable `device_id` a des valeurs manquantes (NaN). Regardons à quoi elles correspondent.

In [ ]:
nan_dev = mapping[mapping["device_id"].isna()]
nan_dev.head()

Les valeurs manquantes semblent concerner les lignes où `customer_id` est inconnu, vérifions si c'est le seul cas :

In [ ]:
print("customer_id des lignes avec device_id manquant :")
print(nan_dev["customer_id"].value_counts(dropna=False))

unknown_cust = mapping[mapping["customer_id"] == "unknown"]
print("\ndevice_id des lignes avec customer_id unknown :")
print(unknown_cust["device_id"].value_counts(dropna=False))

_______________
|>>

Toutes les valeurs manquantes de `device_id` correspondent à un `customer_id` inconnu, et inversement.  
Ces lignes ne relient donc aucun client à une box TV : elles sont inutilisables pour l'attribution.

|<<
_______________


### Identifiants "unknown" :


Dans toutes les tables, l'absence d'identifiant est parfois codée par le texte `"unknown"`. Pandas ne le voit pas comme un manquant : il faut le compter explicitement.

In [ ]:
def count_unknown(datasets):
    id_cols = ["customer_id", "dsp_id", "device_id"]
    results = []
    for name, df in datasets.items():
        for col in id_cols:
            if col in df.columns:
                n_unknown = df[col].astype("string").str.strip().str.lower().eq("unknown").sum()

                results.append({
                    "Table": name,
                    "Variable": col,
                    "Nombre de unknown": n_unknown,
                    "Pourcentage (%)": round(n_unknown / len(df) * 100,2) if len(df) > 0 else 0
                })
    return pd.DataFrame(results)

unknown_summary = count_unknown(datasets)
print(unknown_summary)

Pour les tables `retailer`, `programmatic_pub`, `tv_pub` et `socio_demo`, une ligne "unknown" ne peut pas servir à relier deux tables : à la jointure, tous les "unknown" seraient considérés comme une seule personne, ce qui ferait exploser le nombre de lignes et attribuerait des ventes à des individus qui n'existent pas.

- Pour les totaux : ces lignes sont conservées, l'argent a bien été dépensé.
- Pour toute analyse par client : ces lignes sont exclues.


Une ligne du `mapping` sert à relier un client à ses identifiants média. Elle est donc utile seulement si le customer_id est connu :
- customer_id unknown : ligne inutilisable pour relier une pub à un achat. On l’exclut.
- dsp_id unknown (client connu) : exposition programmatique non mesurable.
- device_id unknown (client connu) : exposition TV non mesurable.

("Non mesurable" ne veut pas dire "non exposé".)

____________
|>>

### Valeurs nulles dans `sales` et `quantity` :

In [ ]:
print(f"sales == 0            : {(orders['sales'] == 0).sum()}")
print(f"quantity == 0         : {(orders['quantity'] == 0).sum()}")

nulle = orders[(orders["sales"]==0) | (orders["quantity"]==0)].copy()
# 2. Poids : part des lignes Order concerné
print(f"Part des lignes Order : {len(nulle) /len(orders):.2%}")

print("\n")
cas = pd.crosstab(nulle["sales"] ==0, nulle["quantity"] ==0,rownames=["sales == 0"], colnames=["quantity == 0"])
cas = cas.reindex(
    index=[False, True],
    columns=[False, True],
    fill_value=0
)
print(cas)

# 3. Quels produits / marques ?
print("\n")
print(nulle["product_name"].value_counts().head(10))
print("\n")
print(nulle.merge(orders[["customer_id", "timestamp_utc", "product_name", "brand"]].drop_duplicates(), how="left")["brand"].value_counts(dropna=False))

# 4. Concentrées dans le temps ou chez certains clients ?
print("\n")
nulle["timestamp_utc"] = pd.to_datetime(nulle["timestamp_utc"])
print(nulle["timestamp_utc"].dt.to_period("M").value_counts().sort_index())
print("\n")
print("Clients  distincts  :",  nulle["customer_id"].nunique(), "pour", len(nulle), "lignes")
print("\n")
print(nulle.groupby("customer_id").size().describe())
print("\n")
print(nulle.groupby("customer_id").size().value_counts().sort_index())

Le poids est faible, cela représente moins de 1% des lignes de commande des clients identifiés. Ces anomalies ne semblent ni être concetrées sur une marque, ni sur un produit donc ça semble être des erreurs / des bruits de collecte. Lorsque quantity = 0 et sales = 0, on supprime ces lignes, elles n’apportent aucune information.

In [ ]:
offerts = orders[(orders["sales"] ==0) & (orders["quantity"] >0)].copy()
normales = orders[(orders["sales"] >0) & (orders["quantity"] >0)].copy()

# Prix de référence par produit, calculé sur les lignes normales
normales["pu"] = normales["sales"] / normales["quantity"]
prix_ref = normales.groupby("product_name")["pu"].median().rename("prix_ref")

# Quantités : ressemblent-elles aux lignes normales ?
print(pd.DataFrame({"A": offerts["quantity"].describe(), "réf": normales["quantity"].describe()}).round(2))

# Produits et marques concernés, par rapport à leur poids habituel
poids = pd.DataFrame({"part dans A (%)": offerts["product_name"].value_counts(normalize=True) *100,"part dans réf (%)": normales["product_name"].value_counts(normalize=True) *100,}).sort_values("part dans A (%)", ascending=False).head(10).round(2)
print("\n")
print(poids)
print("\n")
print(offerts["brand"].value_counts(dropna=False, normalize=True).round(3))

# Clients et temps
print("Clients   distincts   :",   offerts["customer_id"].nunique(), "pour", len(offerts), "lignes")
offerts["timestamp_utc"] = pd.to_datetime(offerts["timestamp_utc"])
print(offerts["timestamp_utc"].dt.to_period("M").value_counts().sort_index())

# Ligne seule ou accompagnée d'autres lignes dans la même commande ?
orders["timestamp_utc"] = pd.to_datetime(orders["timestamp_utc"])
taille =orders.groupby(["customer_id", "timestamp_utc"]).size().rename("nb_lignes")
a = offerts.merge(taille.reset_index(), on=["customer_id", "timestamp_utc"])
print("Lignes         par         commande         (cas         A)         :");
print(a["nb_lignes"].value_counts().head())
print("Dont  la  commande  ne  contient  que  cette  ligne  :",  (a["nb_lignes"] ==1).mean().round(3))

Après  étude  je  décide  également  de  supprimer  ces  lignes  pour  les  raisons  suivantes  :  -  elles faussent le comportement d’achat : ça compte comme une vraie commade alors que le montant est nul - elles n’ont pas de snes pour la partie publicité. Un envoi gratuit n’est pas un achat déclenché par une exposition publicitaire. - leur retrait ne coûte rien. Elles représentent moins de 1% des lignes donc le total des ventes ne bouge pas vraiment si on retire ces lignes.6 295 lignes Order (0,4 %) ont un montant nul ; toutes celles dont la quantité est nulle (2 723) en font partie. Les 3 572 restantes ont une quantité de 1 dans la quasi-totalité des cas, sont isolées dans leur commande, et se répartissent uniformément dans le temps (3 252 clients, tous les mois) : aucun motif d’erreur systématique, mais aucun paiement associé. Décision : retrait de toutes ces lignes. Une commande sans montant n’est pas un achat et fausserait le nombre de commandes par client, le taux de réachat et le panier moyen. Leur impact sur le CA est nul.

### Doublons :

In [ ]:
for name, df in datasets.items():
    print(f"{name:<18}: {df.duplicated().sum():>8,} doublons exacts")

On regarde s'il s'agit de vrais doublons ou de lignes volontairement répétées.

**`retailer`**

In [ ]:
dup = retailer[retailer.duplicated(keep=False)]
dup.sort_values(by=retailer.columns.tolist()).head(10)

In [ ]:
print("Lignes impliquées par type d'événement :")
print(dup["event_name"].value_counts())

**`programmatic_pub`**

In [ ]:
programmatic_pub[
    programmatic_pub.duplicated(keep=False)
].sort_values(by=programmatic_pub.columns.tolist()).head(10)

**`tv_pub`**

In [ ]:
tv_pub[
    tv_pub.duplicated(keep=False)
].sort_values(by=tv_pub.columns.tolist()).head(10)

**`mapping`**

In [ ]:
mapping[mapping.duplicated(keep=False)].head()

In [ ]:
mask_vide = (
    mapping["customer_id"].eq("unknown")
    & mapping["dsp_id"].eq("unknown")
    & mapping["device_id"].isna()
)
print("Lignes 'customer unknown + dsp unknown + device NaN' :", mask_vide.sum())
print("Doublons restants hors ces lignes :", mapping[~mask_vide].duplicated().sum())

Pour `retailer`, `programmatic_pub` et `tv_pub`, les doublons sont exacts sur toutes les variables, avec un timestamp identique à la seconde : ils semblent involontaires.

Pour `mapping`, les seuls doublons sont les lignes sans aucune information utile (`customer_id` et `dsp_id` unknown, `device_id` NaN) : on les supprime toutes.

### Synthèse :

| Constat | Décision | Niveau concerné |
|---|---|---|
| `sales` / `quantity` manquants : structurels, seuls les `Order` en ont | Aucune imputation ; filtre `event_name == "Order"` au moment du calcul des KPI de vente | Analyse |
| `brand` / `product_name` manquants | Aucune imputation ; exclus des analyses par marque / produit uniquement | Analyse |
| `device_id` NaN dans `mapping` = `customer_id` unknown | Lignes exclues pour l'attribution | C |
| Identifiants `unknown` | NaN en B ; lignes retirées en C | B puis C |
| Doublons exacts | Supprimés | B |
| Clients connus avec `dsp_id` ou `device_id` manquant | Conservés, marqués « non mesurables » | C |

___________
reprendre ici

## 2) Nettoyage

Ordre des opérations :
1. doublons exacts ;
2. types
3. "unknown" remplacés par NaN.
4. retrait des Order à montant nulle

### Niveau B : tables nettoyées (pour les totaux)

On commence par retirer les doublons.

In [ ]:
def drop_dups(df, name):
    out = df.drop_duplicates(ignore_index=True)
    print(f"{name:<18}: {len(df) - len(out):,} doublons retirés")
    return out

# mapping : on retire d'abord les lignes entièrement vides (customer et dsp unknown, device NaN)
mapping_c = mapping.loc[~mask_vide]
print(f"{'mapping':<18}: {mask_vide.sum():,} lignes entièrement vides retirées")

retailer_c = drop_dups(retailer, "retailer")
prog_c = drop_dups(programmatic_pub, "programmatic_pub")
tv_c = drop_dups(tv_pub, "tv_pub")
mapping_c = drop_dups(mapping_c, "mapping")
socio_c = socio_demo.copy()

Ensuite on convertit les types de variable et on ordonne les modalités d'âge et de revenu.

In [ ]:
# Types : dates, coût converti (1 milli-cent = 1/100 000 de l'unité monétaire), catégories
for df in (retailer_c, prog_c, tv_c):
    df["timestamp_utc"] = pd.to_datetime(df["timestamp_utc"])

for df in (prog_c, tv_c):
    df["cost"] = df["cost_milli_cent"] / 100_000    # conversion en dollars

for df, cols in [(retailer_c, ["event_name", "brand"]),
                 (prog_c, ["campaign_name", "device_type"]),
                 (socio_c, ["breed"])]:
    df[cols] = df[cols].astype("category")

ordre_age = ["[18-25[", "[25-35[", "[35-45[", "[45-55[", "[55-65[", "[65+["]
ordre_revenu = ["[0-40K$[", "[40-80K$[", "[80-120K$[", "[120-200K$[", "[200K$+["]
socio_c["age"] = pd.Categorical(socio_c["age"], categories=ordre_age, ordered=True)
socio_c["income"] = pd.Categorical(socio_c["income"], categories=ordre_revenu, ordered=True)
assert socio_c["age"].notna().all() and socio_c["income"].notna().all()   # aucune modalité oubliée

Puis on remplace les "unknown" par NaN :

In [ ]:
for df, cols in [(retailer_c, ["customer_id"]),
                 (prog_c, ["dsp_id"]),
                 (tv_c, ["device_id"]),
                 (socio_c, ["customer_id"]),
                 (mapping_c, ["customer_id", "dsp_id", "device_id"])]:
    for c in cols:
        df[c] = df[c].mask(df[c] == "unknown")

Pour finir on retire les lignes des Order à montant nul (quantity == 0 implique sales == 0, une seule règle suffit) :

In [ ]:
masque_zero = (retailer_c["event_name"] == "Order") & (retailer_c["sales"] == 0)
print(f"retailer : {masque_zero.sum():,} lignes Order à montant nul retirées")
retailer_c = retailer_c.loc[~masque_zero]

### Niveau C : tables identifiées (pour les analyses par client)

On retire les lignes dont l'identifiant est manquant. 
Dans `mapping`, on ne retire que les lignes sans `customer_id` : un client connu avec `dsp_id` ou `device_id` manquant est conservé.

In [ ]:
retailer_id = retailer_c.dropna(subset=["customer_id"])
prog_id = prog_c.dropna(subset=["dsp_id"])
tv_id = tv_c.dropna(subset=["device_id"])
socio_id = socio_c.dropna(subset=["customer_id"])

mapping_id = mapping_c.dropna(subset=["customer_id"]).copy()
mapping_id["tv_mesurable"] = mapping_id["device_id"].notna()
mapping_id["dsp_mesurable"] = mapping_id["dsp_id"].notna()

print("Part de clients mesurables :")
print(mapping_id[["tv_mesurable", "dsp_mesurable"]].mean().round(3))

### Tables dérivées, définies une seule fois

In [ ]:
orders_c = retailer_c.loc[retailer_c["event_name"] == "Order"].copy()     # totaux, saisonnalité
orders_id = retailer_id.loc[retailer_id["event_name"] == "Order"].copy()  # analyses par client
orders_id_unique = (orders_id.groupby(["customer_id", "timestamp_utc"])
                .agg(CA=("sales", "sum"),
                     quantite=("quantity", "sum"),
                     nb_lignes=("sales", "size"))
                .reset_index())   # une ligne par commande

customer_order = (orders_id.groupby("customer_id")
          .agg(CA=("sales", "sum"), quantite=("quantity", "sum"),
               nb_lignes=("sales", "size"), nb_produits=("product_name", "nunique"))
          .join(orders_id_unique.groupby("customer_id")
                .agg(nb_commandes=("CA", "size"),
                     premiere_commande=("timestamp_utc", "min"),
                     derniere_commande=("timestamp_utc", "max")))
          .reset_index())
          
customer = (retailer_id[["customer_id"]].drop_duplicates()
           .merge(socio_id, on="customer_id", how="left", validate="one_to_one")
           .merge(customer_order, on="customer_id", how="left", validate="one_to_one"))
cols_achat = ["CA", "quantite", "nb_lignes", "nb_produits", "nb_commandes"]
customer[cols_achat] = customer[cols_achat].fillna(0)
customer["acheteur"] = customer["nb_commandes"] > 0

# regarder les lignes suivantes, est-ce qu'elles sont à faire ? Est-ce qu'elles sont faites ailleurs ?
customer["panier_moyen"] = customer["CA"] / customer["nb_commandes"].replace(0, np.nan)
assert customer["customer_id"].is_unique and customer["age"].notna().all()


## 3) Contrôle des volumes

Nombre de lignes à chaque niveau, pour observer ce qui a été retiré.

In [ ]:
controle = pd.DataFrame({
    "A - brut": [len(retailer), len(programmatic_pub), len(tv_pub), len(socio_demo), len(mapping)],
    "B - nettoyé": [len(retailer_c), len(prog_c), len(tv_c), len(socio_c), len(mapping_c)],
    "C - identifié": [len(retailer_id), len(prog_id), len(tv_id), len(socio_id), len(mapping_id)],
}, index=["retailer", "programmatic_pub", "tv_pub", "socio_demo", "mapping"])

controle["Retiré A -> C (%)"] = ((1 - controle["C - identifié"] / controle["A - brut"]) * 100).round(2)
controle

_____________
|> ça devrait aller ailleurs

In [ ]:
# Période couverte par chaque table (doit être cohérente entre ventes et médias)
for name, df in [("retailer", retailer_c), ("programmatic_pub", prog_c), ("tv_pub", tv_c)]:
    print(f"{name:<18}: {df['timestamp_utc'].min()}  ->  {df['timestamp_utc'].max()}")

# Couverture de socio_demo
print("\nClients de retailer présents dans socio_demo :",
      retailer_id["customer_id"].isin(socio_id["customer_id"]).mean().round(4))

_______________

## 4) Comprendre le marché

### Marché

Ici nous étudions uniquement les données de la table retailer de niveau B.

!!!!!!!  
retailer_c est modifiée dans la partie 4 (le fillna("Non renseignée")), alors que retailer_id, orders_c et orders_id en ont déjà été tirées.
Ces trois tables gardent donc des NaN.
Résultat : « Non renseignée » apparaît dans marques, mais pas dans ca_marque, et pas non plus dans par_marque quand on relance tout proprement.
En plus, ça contredit ta décision de l’audit (« exclus des analyses par marque »).
Choisis une option : soit tu supprimes ce fillna, soit tu le déplaces dans le nettoyage, avant de créer le niveau C.
!!!!!!!

In [ ]:
print(retailer_c.describe(include="all").round(2))

# --- Période d'étude ---
debut, fin = retailer_c["timestamp_utc"].min(), retailer_c["timestamp_utc"].max()
print(f"\nPériode : {debut} -> {fin} ({(fin - debut).days + 1} jours)")

# --- Répartition des marques et nombre de produits par marque ---
repartition = (retailer_c["brand"].value_counts(dropna=False, normalize=True)*100).round(0).astype(int)
produits = retailer_c.groupby("brand", observed=True, dropna=False)["product_name"].nunique(dropna=False)
tableau = pd.concat([
    repartition.rename("% des lignes"),
    produits.rename("Produits distincts")
], axis=1)
print(f"\n{tableau}")

# --- Nombre de clients (identifiés uniquement) ---
print(f"\nClients identifiés        : {retailer_id['customer_id'].nunique():,}")
print(f"Lignes sans client connu  : {retailer_c['customer_id'].isna().sum():,}")

# --- Répartition des types d'événements ---
print(f"\n{(retailer_c["event_name"].value_counts(normalize=True)*100).round(0).astype(int)}")

# --- CA et quantité totaux (événements Order uniquement) ---
print(f"\nCA total        : {orders_c['sales'].sum():,.0f}")
print(f"\nQuantité totale : {orders_c['quantity'].sum():,.0f}")

**Récapitulatif :**
- Période étudiée : du 1er janvier 2024 au 30 juin 2024 (182 jours = 6 mois)  
- 3 marques étudiées : Science Diet (298 produits), Prescription Diet (189 produits) et Hills (18 produits)  
- Nombre de clients identifiés : 1 354 583  
- 3 types d'évènements : Product Page View (vue de la page produit), Add to cart (ajout au panier) et Order (achat)  
- CA total : 62 427 795 $  
- Quantité totale : 1 552 299 produits achetés

_____________
|> c'est des +

In [ ]:

# --- Nombre de clients (identifiés uniquement) ---
print(f"Clients ayant commandé    : {orders_c['customer_id'].nunique():,}")


# --- Répartition des marques (lignes par événement + CA des commandes) ---
marques = pd.crosstab(retailer_c["brand"], retailer_c["event_name"], margins=True, margins_name="Total")
marques["Part des lignes (%)"] = (marques["Total"] / marques.loc["Total", "Total"] * 100).round(0).astype(int)
# Part des commandes de chaque marque parmi toutes les commandes
marques["Part des orders (%)"] = (marques["Order"] / marques.loc["Total", "Order"] * 100).round(0).astype(int)
# Part des commandes parmi tous les événements de chaque marque
marques["Orders / marque (%)"] = (marques["Order"] / marques["Total"] * 100).round(0).astype(int)
print(marques)

ca_marque = orders_c.groupby("brand", observed=True)["sales"].agg(commandes="count", CA="sum")
ca_marque["Part du CA (%)"] = (ca_marque["CA"] / ca_marque["CA"].sum() * 100).round(0).astype(int)
print(ca_marque)

La période étudiée est du 1er janvier 2024 au 30 juin 2024, soit une période de 6 mois.
Le table de données contient 1 354 583 clients identifiés, parmis ceux-là, 502 025 ont passé commande.

Environ 2 tiers des lignes correspondent seulement à une vue de la page du produit. 21% sont des ajouts au paniers et 15% un achat.

Il existe 3 marques : Hills, Prescription Diet et Science Diet. La majorité des lignes concernent Science Diet (80%). Cependant c'est la marque Hills qui a le plus d'achat (parmi les actions sur la marque). Mais de part sa faible présence elle représente seulement 2% du CA total.

Sur 505 produits différents, 18 sont de la marque Hills, 189 de la marque Prescription Diet et 298 de la marque Science Diet.

Sur cette période de 6 mois, le chiffre d'affaire total est de 62 427 795 pour 1 555 998 produits vendus.

|<
_____________

#### Temporalité

Observons maintenant de plus près la temporalité.

In [ ]:
# --- Préparation ---
orders_c["mois"] = orders_c["timestamp_utc"].dt.to_period("M")
orders_c["jour"] = orders_c["timestamp_utc"].dt.date
orders_c["jour_sem"] = orders_c["timestamp_utc"].dt.dayofweek
orders_c["heure"] = orders_c["timestamp_utc"].dt.hour

jours_fr = ["Lun", "Mar", "Mer", "Jeu", "Ven", "Sam", "Dim"]

In [ ]:
def agg(col):
    """CA et nombre de commandes par modalité de `col`."""
    return orders_c.groupby(col)["sales"].agg(CA="sum", commandes="count")


def tracer(df, titre, xlabel, kind="bar", rotation=0,
           ylim_ca=None, ylim_commandes=None):
    """Deux panneaux : CA puis nombre de commandes, avec zoom optionnel sur Y."""
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    for ax, col, ylabel, ylim in zip(
        axes,
        ["CA", "commandes"],
        ["CA", "Nombre de commandes"],
        [ylim_ca, ylim_commandes]
    ):
        df[col].plot(kind=kind, ax=ax, color="#3b6ea5")
        ax.set_title(f"{ylabel} {titre}")
        ax.set_xlabel(xlabel)
        ax.set_ylabel(ylabel)
        ax.tick_params(axis="x", rotation=rotation)
        ax.grid(axis="y", alpha=0.3)

        if ylim is not None:
            ax.set_ylim(ylim)

    if xlabel == "Heure":
        for ax in axes:
            ax.set_xticks(range(24))
            ax.set_xticklabels([f"{h}h" for h in range(24)], rotation=45)
            
    plt.tight_layout()
    plt.show()

##### Par mois :

In [ ]:
par_mois = agg("mois")
tracer(par_mois, "par mois", "Mois", kind="bar")

In [ ]:
tracer(par_mois, "par mois", "Mois", kind="bar", ylim_ca=(0.95e7, 1.1e7), ylim_commandes=(220000,250000))

Que ce soit en terme de CA ou en nombre de commandes, on observe un creu en avril et des pics en janvier, février et mai.

##### Par jour :

In [ ]:
# --- Par jour (série quotidienne + moyenne mobile sur 7 jours) ---
par_jour = agg("jour")
par_jour.index = pd.to_datetime(par_jour.index)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for ax, col in zip(axes, ["CA", "commandes"]):
    ax.plot(par_jour.index, par_jour[col], alpha=0.4, label="Quotidien")
    ax.plot(par_jour.index, par_jour[col].rolling(7, center=True).mean(), label="Moyenne mobile 7 j")
    ax.set_title(f"{col} par jour")
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

##### Par jour de la semaine :

In [ ]:
# --- Par jour de la semaine ---
par_js = agg("jour_sem")
par_js.index = jours_fr
tracer(par_js, "par jour de la semaine", "Jour", kind="bar")

In [ ]:
tracer(par_js, "par jour de la semaine", "Jour", kind="bar", ylim_ca=(8e6,9.5e6),ylim_commandes=(195000,210000))

On observe un creu le week-end (samedi et dimanche).

##### Par heure de la journée

In [ ]:
# --- Par heure de la journée ---
par_heure = agg("heure")
tracer(par_heure, "par heure", "Heure", kind="line")

L'heure est en UTC car le fuseau horaire des clients n'est pas fourni. Le creux étant à 8-10h UTC, c'est cohérent avec la uit aux États-Unis. Cependant cela reste une hypothèse.

Si on fait l'hypothèse que c'est le fuseau horaire de la côte Est des États-Unis, cela donne le graphique suivant.

In [ ]:
# Convertir UTC en heure locale de New York
orders_c["timestamp_ny"] = (
    pd.to_datetime(orders_c["timestamp_utc"], utc=True)
      .dt.tz_convert("America/New_York")
)
orders_c["heure_ny"] = orders_c["timestamp_ny"].dt.hour
# --- Par heure de la journée ---
par_heure_ny = agg("heure_ny")
tracer(par_heure_ny, "par heure", "Heure", kind="line")


Dans le cas de cette hypothèse, on observe un gros creu dans la nuit, deux pics vers 10h et 19h et à nouveau un petit creu en journée, aux alentours de 15h.

______________
revoir ça

In [ ]:
# --- Bonus : panier moyen par ligne de commande selon le mois et le jour de la semaine ---
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
(par_mois["CA"] / par_mois["commandes"]).plot(kind="bar", ax=axes[0], color="#3b6ea5")
axes[0].set_title("CA moyen par ligne de commande, par mois")
(par_js["CA"] / par_js["commandes"]).plot(kind="bar", ax=axes[1], color="#3b6ea5")
axes[1].set_title("CA moyen par ligne de commande, par jour de la semaine")
for ax in axes:
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

_________________

#### Comportement d'achat

In [ ]:
# --- Une ligne par commande (même client, même timestamp) ---
print(f"{len(orders_id):,} lignes Order -> {len(orders_id_unique):,} commandes")
print(orders_id_unique["nb_lignes"].value_counts().head())

Un client peut apparaitre plusieurs fois dans la base de données retailer_id. S'il a plusieurs lignes à la même heure, je prends comme hypothèse qu'il a avait des produits différents dans son panier mais que cela correspond à une unique commande. Je crée donc une nouvelle base de données avec une ligne par commande afin d'étudier le comportement d'achat.

In [ ]:
# Identifiants des commandes contenant exactement 2 lignes
commandes_2 = orders_id_unique.loc[orders_id_unique["nb_lignes"] == 2].copy()

print(f"Nombre de commandes à 2 lignes : {len(commandes_2):,}")

# Récupérer les lignes d'origine appartenant aux commandes à 2 lignes
lignes_2 = (
    orders_id.merge(
        commandes_2[["customer_id", "timestamp_utc"]],
        on=["customer_id", "timestamp_utc"],
        how="inner"
    )
    .sort_values(["customer_id", "timestamp_utc"])
)

# Afficher quelques exemples
cols = [
    "customer_id", "timestamp_utc", "brand",
    "product_name", "sales", "quantity"
]

display(lignes_2[cols].head(20))

In [ ]:
customer_order["panier_moyen"] = customer_order["CA"] / customer_order["nb_commandes"]
print(f"Acheteurs identifiés : {len(customer_order):,}")
print(customer_order[["CA", "quantite", "nb_produits", "nb_commandes", "panier_moyen"]]
      .describe(percentiles=[.25, .5, .75, .9, .99]).round(2))

In [ ]:
# --- Taux de réachat : part des acheteurs avec au moins 2 commandes ---
taux_reachat = (customer_order["nb_commandes"] >= 2).mean()
print(f"Taux de réachat (>= 2 commandes) : {taux_reachat:.1%}")

# Variante : au moins 2 jours d'achat distincts (plus robuste si plusieurs commandes à la suite)
jours_achat = orders_id_unique.assign(jour=orders_id_unique["timestamp_utc"].dt.date).groupby("customer_id")["jour"].nunique()
print(f"Taux de réachat (>= 2 jours distincts) : {(jours_achat >= 2).mean():.1%}")

repartition = customer_order["nb_commandes"].clip(upper=5).value_counts().sort_index()
repartition.index = [str(i) if i < 5 else "5+" for i in repartition.index]
print(pd.DataFrame({"Clients": repartition, "Part (%)": (repartition / repartition.sum() * 100).round(1)}))

#### Etude du parcours sur le site
On distingue les 3 étapes du parcours d'un client sur le site : voir un produit, l’ajouter au panier, commander. Ces trois étapes sont dans event_name : Product Page View, Add to cart, Order.
Observons le taux de conversion, qui est la part de ceux qui passent à l’étape suivante.

In [ ]:
etapes = ["Product Page View", "Add to cart", "Order"]

funnel = pd.DataFrame({
    "Lignes": retailer_c["event_name"].value_counts().reindex(etapes),
    "Clients uniques": retailer_id.groupby("event_name", observed=True)["customer_id"].nunique().reindex(etapes),
})
for col in ["Lignes", "Clients uniques"]:
    funnel[f"{col} : % de l'étape précédente"] = (funnel[col] / funnel[col].shift(1) * 100).round(1)
    funnel[f"{col} : % de la 1re étape"] = (funnel[col] / funnel[col].iloc[0] * 100).round(1)
funnel

Il y a une subtilité : tous les acheteurs n’ont pas forcément une vue enregistrée dans la période. Vérifie-le, car ça change la lecture des pourcentages :

In [ ]:
vus = set(retailer_id.loc[retailer_id["event_name"] == "Product Page View", "customer_id"])
paniers = set(retailer_id.loc[retailer_id["event_name"] == "Add to cart", "customer_id"])
acheteurs = set(retailer_id.loc[retailer_id["event_name"] == "Order", "customer_id"])

print(f"Vu -> panier (parmi les clients ayant vu)   : {len(vus & paniers) / len(vus):.1%}")
print(f"Panier -> commande (parmi ceux au panier)  : {len(paniers & acheteurs) / len(paniers):.1%}")
print(f"Vu -> commande : {len(vus & acheteurs) / len(vus):.1%}")
print(f"Acheteurs sans aucune vue enregistrée       : {len(acheteurs - vus):,} ({len(acheteurs - vus) / len(acheteurs):.1%})")

- Moins d’un client sur deux qui a consulté un produit en a ajouté un au panier sur la période. C’est l’endroit où l’on perd le plus de monde.
- Huit clients sur dix qui ont mis un produit au panier ont fini par commander. Une fois qu’un client a ajouté un produit au panier, il achète presque toujours.

6,0 % des acheteurs (30 127) n’ont aucune vue enregistrée. C’est peu, mais ils sont hors de l’entonnoir tel que tu le comptes.

Attention cependant. 
- Ce sont des taux par client sur six mois, pas par visite. « 80 % » veut dire « 80 % des clients qui ont un jour ajouté un produit au panier ont un jour commandé », et non « 80 % des paniers aboutissent ». Rien dans les données ne prouve que le panier précède la commande dans le temps. N’écris donc pas « 80 % des paniers sont convertis ».
- Les 6 % d’acheteurs sans vue. Je vois deux explications possibles : la vue a eu lieu avant le 1er janvier, car les données commencent à cette date, ou le client est arrivé sans passer par une fiche produit (recommande directe, lien, application). Ce sont des hypothèses. Tu peux tester la première :

In [ ]:
sans_vue = acheteurs - vus
premiere_cmd = customer_order.set_index("customer_id")["premiere_commande"]
t = premiere_cmd.dt.to_period("M").to_frame("mois")
t["sans_vue"] = t.index.isin(sans_vue)
print((t.groupby("mois")["sans_vue"].mean() * 100).round(1))

La part n'est pas plus haute en janvier qu'en juin (c'est l'inverse), elle est plutôt stable, l'explication est plutôt un parcours qui ne passe pas par la fiche produit


En clients uniques, 44,2 % des clients ayant consulté un produit en ont ajouté un au panier, et 80,1 % de ceux qui ont rempli un panier ont commandé. La principale perte se situe donc entre la consultation et l’ajout au panier. Ces taux sont calculés par client sur six mois, et non par session. 6,0 % des acheteurs n’ont aucune consultation enregistrée : [explication selon ton test]. Ils restent dans le décompte des acheteurs, mais pas dans les taux ci-dessus.

conversion par marque. Le principe est de compter, pour chaque marque, les clients uniques à chaque étape.

In [ ]:
par_marque = (retailer_id.groupby(["brand", "event_name"], observed=True)["customer_id"]
                         .nunique().unstack()[etapes])
par_marque["Vue -> commande (%)"] = (par_marque["Order"] / par_marque["Product Page View"] * 100).round(1)
par_marque["Panier -> commande (%)"] = (par_marque["Order"] / par_marque["Add to cart"] * 100).round(1)
print(par_marque)

Faut-il faire aussi par produit ?

#### Etude de sales et quantity


On commence par créer une nouvelle variable : le prix unitaire.

In [ ]:
# --- Variable prix unitaire (une seule fois) ---
assert (orders_c["quantity"] > 0).all() and (orders_c["sales"] > 0).all()
orders_c["prix_unitaire"] = orders_c["sales"] / orders_c["quantity"]

In [ ]:
print(orders_c[["sales", "quantity", "prix_unitaire"]]
      .describe(percentiles=[.01, .25, .5, .75, .95, .99]).round(2))

print("\nQuantité par ligne (% des lignes) :")
print((orders_c["quantity"].value_counts(normalize=True).sort_index().head(8) * 100).round(2))

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

orders_c["sales"].clip(upper=orders_c["sales"].quantile(0.99)).plot(kind="hist", bins=60, ax=axes[0, 0])
axes[0, 0].set_title("sales par ligne (tronqué au 99e centile)")

np.log10(orders_c["sales"]).plot(kind="hist", bins=60, ax=axes[0, 1])
axes[0, 1].set_title("log10(sales)")

orders_c["prix_unitaire"].clip(upper=orders_c["prix_unitaire"].quantile(0.99)).plot(kind="hist", bins=60, ax=axes[1, 1])
axes[1, 1].set_title("Prix unitaire (tronqué au 99e centile)")

plt.tight_layout()
plt.show()

Poursuivons avec le prix unitaire par produit.

In [ ]:
prix_produit = (orders_c.groupby("product_name")
    .agg(lignes=("sales", "size"), quantite=("quantity", "sum"), CA=("sales", "sum"),
         prix_median=("prix_unitaire", "median"), prix_min=("prix_unitaire", "min"),
         prix_max=("prix_unitaire", "max"), prix_std=("prix_unitaire", "std"))
    .assign(ecart_relatif=lambda d: d["prix_std"] / d["prix_median"]))

print(f"{len(prix_produit)} produits")
print(prix_produit.sort_values("CA", ascending=False).head(10).round(2))

# Le prix est-il stable pour un même produit ?
print("\nÉcart-type / médiane, par produit :")
print(prix_produit["ecart_relatif"].describe(percentiles=[.5, .9, .99]).round(3))

# Un produit appartient-il à une seule marque ?
print("\nProduits rattachés à plusieurs marques :",
      (orders_c.groupby("product_name")["brand"].nunique() > 1).sum())

In [ ]:
orders_c["prix_ref"] = orders_c.groupby("product_name")["prix_unitaire"].transform("median")
orders_c["ecart_prix"] = orders_c["prix_unitaire"] / orders_c["prix_ref"] - 1

e = orders_c["ecart_prix"].dropna()
print(f"Lignes à ±1 % du prix de référence : {(e.abs() <= 0.01).mean():.1%}")
print(f"Prix inférieur de plus de 10 %       : {(e < -0.10).mean():.1%}")
print(f"Prix supérieur de plus de 10 %       : {(e > 0.10).mean():.1%}")

Si presque toutes les lignes sont à ±1 %, le prix est un prix catalogue, et les variations de CA viennent des volumes. Si une part notable est nettement en dessous, c’est probablement des promotions, ce qui peut brouiller la partie 2 : une hausse des ventes peut venir d’une promo et non de la publicité. C’est une hypothèse, qu’il faudra confirmer par les dates.

évolution dans le temps et lignes atypiques.

In [ ]:
# Prix médian mensuel des 5 produits qui font le plus de CA
top = prix_produit.nlargest(5, "CA").index
evol = (orders_c[orders_c["product_name"].isin(top)]
        .assign(mois=lambda d: d["timestamp_utc"].dt.to_period("M"))
        .groupby(["mois", "product_name"])["prix_unitaire"].median().unstack())
evol.plot(figsize=(10, 4), marker="o")
plt.title("Prix unitaire médian par mois (top 5 produits en CA)")
plt.ylabel("Prix unitaire"); plt.grid(alpha=0.3); plt.show()

# Gros achats
gros = orders_c[orders_c["quantity"] >= 10]
print(f"Lignes avec quantité >= 10 : {len(gros):,} ({len(gros) / len(orders_c):.2%} des lignes, "
      f"{gros['sales'].sum() / orders_c['sales'].sum():.2%} du CA)")
print(orders_c.nlargest(10, "sales")[["customer_id", "product_name", "sales", "quantity", "prix_unitaire"]])

# Prix par marque
print(orders_c.groupby("brand", observed=True)["prix_unitaire"].describe().round(2))

#### Description du profil des clients

**Question** : qui sont les clients du retailer, et quels profils achètent le plus ?

**Méthode** : la table `customer` (définie en 2.3) contient tous les clients identifiés du retailer, acheteurs ou non, reliés à `socio_demo` par une jointure 1 pour 1. Un client est dit *acheteur* s'il a au moins une commande sur la période. Pour chaque variable (race de l'animal, âge, revenu), on regarde trois choses : le poids du profil dans la clientèle, son **taux d'achat** (part de clients qui achètent) et son **comportement d'achat** (CA, commandes, panier).

In [ ]:
print(f"Clients : {len(customer):,}")
print(f"Acheteurs : {customer['acheteur'].sum():,} ({customer['acheteur'].mean():.0%})")
print("Clients retailer absents de socio_demo :", (~customer["breed"].notna()).sum())

La base compte **1 354 524 clients**, tous présents dans `socio_demo` : le profil est connu pour 100 % d'entre eux. **500 877 clients (37,0 %) ont commandé** au moins une fois ; les autres ont seulement consulté des produits ou ajouté au panier.

In [ ]:
variables_socio = ["breed", "age", "income"]
taux_global = customer["acheteur"].mean() * 100

def profil(var):
    g = customer.groupby(var, observed=True)
    t = pd.DataFrame({
        "Clients": g.size(),
        "Acheteurs": g["acheteur"].sum(),
        "CA": g["CA"].sum(),
        "Commandes": g["nb_commandes"].sum(),
    })
    t["Part clients (%)"]   = t["Clients"]   / t["Clients"].sum()   * 100
    t["Part acheteurs (%)"] = t["Acheteurs"] / t["Acheteurs"].sum() * 100
    t["Taux d'achat (%)"]   = t["Acheteurs"] / t["Clients"] * 100
    t["Part du CA (%)"]     = t["CA"] / t["CA"].sum() * 100
    t["CA par client"]      = t["CA"] / t["Clients"]
    t["CA par acheteur"]    = t["CA"] / t["Acheteurs"]
    t["Commandes par acheteur"] = t["Commandes"] / t["Acheteurs"]
    t["Panier moyen"]       = t["CA"] / t["Commandes"]
    return t

cols_qui = ["Clients", "Part clients (%)", "Acheteurs", "Part acheteurs (%)", "Taux d'achat (%)"]
cols_comportement = ["Part du CA (%)", "CA par client", "CA par acheteur", "Commandes par acheteur", "Panier moyen"]

def graphe_taux(t, titre, ax):
    t["Taux d'achat (%)"].plot(kind="bar", ax=ax, color="steelblue", rot=0)
    ax.axhline(taux_global, color="red", ls="--", label=f"moyenne : {taux_global:.1f} %")
    ax.set_title(titre); ax.set_xlabel(""); ax.set_ylabel("Taux d'achat (%)")
    ax.set_ylim(0, None); ax.legend(); ax.grid(axis="y", alpha=0.3)

**Race de l'animal**

In [ ]:
t_breed = profil("breed")
display(t_breed[cols_qui].round(1))
display(t_breed[cols_comportement].round(1))
fig, ax = plt.subplots(figsize=(5, 4)); graphe_taux(t_breed, "Taux d'achat selon la race", ax); plt.show()

61 % des clients sont de race pure (*Purebred*) et 39 % sont croisés (*Mixed-breed*). Les deux groupes achètent à peu près autant (taux d'achat 36 % et 38 %) et dépensent des montants proches par acheteur (125 $ contre 118 $). La race **différencie peu** les clients.

*(Hypothèse : `breed` désigne la race de l'animal, ce qui est cohérent avec un retailer d'alimentation pour animaux ; le dictionnaire de données ne le précise pas.)*

**Age**

In [ ]:
t_age = profil("age")
display(t_age[cols_qui].round(1))
display(t_age[cols_comportement].round(1))
fig, ax = plt.subplots(figsize=(7, 4)); graphe_taux(t_age, "Taux d'achat selon l'âge", ax); plt.show()

La clientèle est répartie de façon assez uniforme entre tranches d'âge (13 à 19 % chacune). Cependant, l'âge semble différencier les clients :
- le **taux d'achat** suit une courbe en cloche : 30 % chez les 18-25 ans, 45 % chez les 35-45 ans (pic), puis retour à 30 % après 65 ans ;
- le **CA par client** suit la même courbe (de 37 $ à 58 $) ;
- le **CA par acheteur** varie peu (117 à 129 $) : l'écart vient donc surtout du nombre de clients qui achètent, pas de ce qu'ils dépensent une fois qu'ils achètent.

**Revenu**

In [ ]:
t_inc = profil("income")
display(t_inc[cols_qui].round(1))
display(t_inc[cols_comportement].round(1))
fig, ax = plt.subplots(figsize=(7, 4)); graphe_taux(t_inc, "Taux d'achat selon le revenu", ax); plt.show()

Le revenu joue moins que l'âge : le taux d'achat reste entre 34 % et 40 %, et chaque tranche pèse 18 à 21 % du CA. La relation n'est pas monotone :
- la tranche la plus modeste (< 40 K$) achète le plus souvent (40 %) et a le CA par acheteur le plus élevé (139 $, contre 114 $ pour 120-200 K$) ;
- la tranche ≥ 200 K$ achète le moins (34 %).

Le panier moyen est proche entre tranches (44 à 47 $) car les acheteurs à faible revenu passent plus de commandes. Le revenu n'est donc pas un moteur d'achat évident.

**Age x revenu**

In [ ]:
taux_croise = pd.crosstab(customer["age"], customer["income"],
                          values=customer["acheteur"].astype(int), aggfunc="mean") * 100
acheteurs_croise = pd.crosstab(customer["age"], customer["income"],
                               values=customer["acheteur"].astype(int), aggfunc="sum")

plt.figure(figsize=(9, 5))
sns.heatmap(taux_croise, annot=True, fmt=".1f", cmap="Blues", cbar_kws={"label": "Taux d'achat (%)"})
plt.title("Taux d'achat (%) selon l'âge et le revenu"); plt.xlabel("Revenu"); plt.ylabel("Âge")
plt.tight_layout(); plt.show()

print(f"Taux min : {taux_croise.min().min():.1f} %  |  max : {taux_croise.max().max():.1f} %")
print(f"Acheteurs dans la plus petite case : {int(acheteurs_croise.min().min()):,}")

Le taux d'achat va de **27,0 %** (18-25 ans, ≥ 200 K$) à **49,3 %** (35-45 ans, < 40 K$). La plus petite case compte plus de 9 000 acheteurs donc ces écarts ne viennent pas de petits effectifs. Dans chaque tranche d'âge, on retrouve la même forme selon le revenu : les < 40 K$ et 80-120 K$ achètent le plus, les 40-80 K$ et surtout les ≥ 200 K$ le moins. Les effets de l'âge et du revenu s'additionnent donc plutôt qu'ils ne s'opposent.

**Conséquence pour la partie 2.** Le profil influence la probabilité d'acheter indépendamment de toute publicité. Comparer exposés et non-exposés sans tenir compte de l'âge et du revenu mélangerait l'effet de la pub et celui de la composition des deux groupes (*biais de confusion*).

##### Synthèse : profil des clients

- **Base** : 1 354 524 clients identifiés, profil socio-démo connu à 100 % ; 500 877 acheteurs (**37,0 %**).
- **Clientèle** : répartie de façon assez uniforme entre tranches d'âge (13-19 %) et de revenu (16-22 %) ; 61 % de race pure, 39 % de croisés.
- **Qui achète le plus** : les **35-45 ans** (45 %) puis les 45-55 ans (41 %) ; côté revenu, les **< 40 K$** (40 %). Profil le plus probable : 35-45 ans à moins de 40 K$ (**49,3 %**).
- **Qui achète le moins** : les **18-25 ans** et les **65+** (30 %) ; côté revenu, les ≥ 200 K$ (34 %). Profil le moins probable : 18-25 ans à ≥ 200 K$ (**27,0 %**).
- **Ce que dépensent les acheteurs** : montants assez homogènes (CA par acheteur 114 à 139 $ selon le profil, panier moyen 44-47 $). Les écarts de CA entre profils viennent surtout du **taux d'achat**, peu du montant dépensé.
- **Peu discriminant** : la race de l'animal (36 % contre 38 %).
- **Limites** : ce sont des associations, pas des causes. Aucun test statistique n'a été fait : avec des effectifs aussi grands, même un petit écart serait « significatif », donc on juge l'ampleur des écarts. 
- **Lien avec la partie 2** : l'âge et le revenu sont à contrôler (comparaison par strates) quand on comparera exposés et non-exposés.

## 5) Comprendre l’impact de la publicité sur les ventes

### Description des tables médias

Deux tables décrivent la publicité diffusée :
- `tv_publisher` : publicités vues sur une box TV (`device_id`) ;
- `programmatic_publisher` : publicités achetées en programmatique et vues par un internaute (`dsp_id`), avec la campagne (`campaign_name`) et le type d'appareil (`device_type`).

**Lecture des tables :**
- une ligne correspond à une impression (une publicité vue une fois) : `cost_milli_cent` est donc le coût d'une impression ;
- 1 milli-cent = 1/1 000 de cent = 1/100 000 $ ;
- `device_id` identifie une box, donc un foyer, et non une personne ;
- `dsp_id` identifie un internaute ;
- `campaign_name` prend deux valeurs : *Contextual*, la publicité s'affiche selon le contenu de la page visitée, quel que soit l'internaute ; *Retargeting*, elle est montrée à des internautes déjà passés sur le site ;
- la modalité `TV` de `device_type` désigne la TV connectée achetée en programmatique, distincte de `tv_publisher`.


In [ ]:
def resume_media(df, id_col):
    sans_id = df[id_col].isna()
    return pd.Series({
        "Impressions": len(df),
        "Budget ($)": df["cost"].sum(),
        "Budget sans id (%)": df.loc[sans_id, "cost"].sum() / df["cost"].sum() * 100,
        "Individus distincts": df[id_col].nunique(),
        "Nombre moyen d'impressions par individu": (~sans_id).sum() / df[id_col].nunique(),
        "Jours d'impressions": df["timestamp_utc"].dt.date.nunique(),
    })

synthese_media = pd.DataFrame({"Programmatique": resume_media(prog_c, "dsp_id"),
                               "TV": resume_media(tv_c, "device_id")})
synthese_media.loc["Part du budget (%)"] = (synthese_media.loc["Budget ($)"]
                                            / synthese_media.loc["Budget ($)"].sum() * 100)
synthese_media.round(1)

**Vue d'ensemble.** Sur la période, 228 k$ ont été dépensés en publicité :
- programmatique : 17,5 M d'impressions pour 93,0 k$ (41 % du budget), diffusées tous les jours du 1er janvier au 30 juin (182 jours) ;
- TV : 5,8 M d'impressions pour 135,4 k$ (59 % du budget), à partir du 29 février. Sur les 123 jours qui séparent le 29 février du 30 juin, 109 seulement comptent des impressions : la diffusion TV s'est arrêtée 14 jours.

La TV représente un quart des impressions mais près de 60 % du budget, elle est plus cher.

**Personnes touchées.** Le programmatique a touché 5,1 M d'internautes (3,1 impressions chacun en moyenne), la TV 2,07 M de box (2,7 impressions en moyenne).

**Budget non attribuable.** 10,9 % du budget programmatique et 2,9 % du budget TV concernent des impressions sans identifiant. Cet argent a été dépensé mais ne pourra jamais être relié à un achat.

______________
|> reprendre le graphique

In [ ]:
# TV : prix fixe ?
print(tv_c["cost_milli_cent"].value_counts())
print(tv_c["cost_milli_cent"].value_counts(normalize=True))


# Programmatique : distribution par campagne et par device
print("\n")
print(prog_c.groupby(["campaign_name", "device_type"], observed=True)["cost_milli_cent"]
      .describe(percentiles=[.5]).round(0).astype(int))

fig, ax = plt.subplots(figsize=(9, 4))
for camp, d in prog_c.groupby("campaign_name", observed=True):
    ax.hist(d["cost_milli_cent"], bins=np.logspace(np.log10(100), np.log10(12000), 80), alpha=.5, label=camp)
ax.set_xscale("log"); ax.set_xlabel("Coût d'une impression (milli-cents, échelle log)")
ax.legend(); plt.show()

**Coût d'une impression.** 
TV : prix fixe (0,023 $ l'impression, 14 exceptions négligeables). Programmatique : prix variable, achat aux enchères. Le retargeting sur PC et la TV connectée sont les plus chers (CPM moyen de 11 à 12 $), le téléphone le moins cher (3 à 4 $). 
Les modalités `Robot` et `Unknown` comptent moins de 700 impressions : négligeables.

|<
___________________

où va l'argent du programmatique

In [ ]:
impr = pd.crosstab(prog_c["device_type"], prog_c["campaign_name"])
impr["Total"] = impr.sum(axis=1)
print("Part des impressions (%)"); print((impr / impr["Total"].sum() * 100).round(1))

budget = prog_c.pivot_table(index="device_type", columns="campaign_name", values="cost",
                            aggfunc="sum", observed=True)
budget["Total"] = budget.sum(axis=1)
print("\nPart du budget (%)");     print((budget / budget["Total"].sum() * 100).round(1))

**Répartition.** Le contextuel fait 78 % des impressions, mais le retargeting, 3 fois plus cher, prend 45 % du budget. Le PC concentre 60 % du budget programmatique.

________________
|> à faire avec l'heure de ny plutôt non ?

calendrier de diffusion

In [ ]:
prog_jour = (prog_c.groupby([pd.Grouper(key="timestamp_utc", freq="D"), "campaign_name"], observed=True)
             .agg(impressions=("cost", "size"), budget=("cost", "sum")).unstack())
tv_jour = tv_c.groupby(pd.Grouper(key="timestamp_utc", freq="D")).agg(
    impressions=("cost", "size"), budget=("cost", "sum"))

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
for ax, mesure in zip(axes, ["impressions", "budget"]):
    for camp in prog_jour[mesure].columns:
        ax.plot(prog_jour.index, prog_jour[(mesure, camp)], label=f"Programmatique - {camp}")
    ax.plot(tv_jour.index, tv_jour[mesure], label="TV")
    ax.axvline(pd.Timestamp("2024-02-29"), color="grey", ls="--")
    ax.set_title(f"{mesure.capitalize()} par jour"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

# Jour de la semaine et heure (UTC)
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for nom, df in [("Programmatique", prog_c), ("TV", tv_c)]:
    (df["timestamp_utc"].dt.dayofweek.value_counts(normalize=True).sort_index()
       .set_axis(jours_fr) * 100).plot(ax=axes[0], marker="o", label=nom)
    (df["timestamp_utc"].dt.hour.value_counts(normalize=True).sort_index() * 100).plot(ax=axes[1], marker="o", label=nom)
axes[0].set_title("Part des impressions par jour (%)"); axes[1].set_title("Part des impressions par heure UTC (%)")
for ax in axes: ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

|<
__________________

combien de fois une même personne voit la pub

In [ ]:
# La fréquence d'exposition : combien d'impressions chaque individu reçoit en moyenne, et quelle est la distribution de ces fréquences ?
freq_prog = prog_id.groupby("dsp_id").size()
freq_tv = tv_id.groupby("device_id").size()
print(pd.DataFrame({"Programmatique": freq_prog.describe(percentiles=[.25, .5, .75, .9, .99]),
                    "TV": freq_tv.describe(percentiles=[.25, .5, .75, .9, .99])}).round(1))

# Qui reçoit quelle(s) campagne(s) ?
combi = prog_id.groupby(["dsp_id", "campaign_name"], observed=True).size().unstack(fill_value=0) > 0
print(combi.value_counts(normalize=True).mul(100).round(1))

**Fréquence.** La moitié des personnes exposées n'a vu qu'une seule pub, et 1 % en a vu plus de 20. Les maximums (194 pour un internaute, 234 pour une box) restent plausibles : 1 à 2 pubs par jour. Les deux campagnes touchent des publics presque séparés : 64 % des personnes exposées n'ont vu que du contextuel, 32 % que du retargeting, 4 % les deux.

**À retenir pour la suite**
- La TV démarre le 29 février : on dispose d'une période « avant » sans TV pour mesurer son effet.
- Le retargeting vise des personnes déjà intéressées, donc il sera analysé à part.
- 1,8 M de personnes ont été retargetées, plus que les 1,35 M clients du site : à vérifier lors de la jointure avec `mapping`.

### Contruction des tables de données

Pour cette deuxième partie nous avons besoin de nouvelles tables de données.

**vérifications**

In [ ]:
print("Doublons dsp_id :", mapping_id["dsp_id"].dropna().duplicated().sum())
print("Doublons device_id :", mapping_id["device_id"].dropna().duplicated().sum())

In [ ]:
print("Retailer")
print("Lignes :", len(retailer_id))
print("Clients distincts :", retailer_id["customer_id"].nunique())

print("\nMapping")
print("Lignes :", len(mapping_id))
print("Clients distincts :", mapping_id["customer_id"].nunique())

In [ ]:
clients_retailer = set(retailer_id["customer_id"].dropna().unique())
clients_mapping = set(mapping_id["customer_id"].dropna().unique())

communs = clients_retailer & clients_mapping
uniquement_retailer = clients_retailer - clients_mapping
uniquement_mapping = clients_mapping - clients_retailer

print("Clients communs :", len(communs))
print("Uniquement dans retailer :", len(uniquement_retailer))
print("Uniquement dans mapping :", len(uniquement_mapping))

print("\nPart des clients retailer présents dans mapping :",
      round(len(communs) / len(clients_retailer) * 100, 1), "%")

print("Part des clients mapping présents dans retailer :",
      round(len(communs) / len(clients_mapping) * 100, 1), "%")

C'est bon, il y a bien + d'id client sur mapping que sur retailer.

#### Table `parcours` : la frise de chaque client

Une ligne = un événement : vue, panier, commande, pub programmatique (par campagne) ou pub TV. Choix :
- une ligne par commande, avec le CA total de la commande ;
- les pubs sont rattachées à un client via `mapping` ; celles qui ne correspondent à aucun client sont retirées (nombre affiché) ;
- si une action sur le site et une pub ont lieu à la même seconde, l'action est placée avant : une pub doit précéder strictement l'action pour avoir pu la provoquer ;
- l'absence de pub dans la frise ne veut pas dire « non exposé » pour un client non mesurable : la règle NaN s'applique dans `pop` et `cmd`.

In [ ]:
ordre_types = ["Product Page View", "Add to cart", "Order", "Pub Contextual", "Pub Retargeting", "Pub TV"]

# Site : vues et paniers (une ligne par événement)
vp = retailer_id.loc[retailer_id["event_name"] != "Order"]
site = pd.DataFrame({
    "customer_id": vp["customer_id"],
    "timestamp_utc": vp["timestamp_utc"],
    "type": vp["event_name"],
})

# Site : commandes (une ligne par commande, CA total)
commandes = pd.DataFrame({
    "customer_id": orders_id_unique["customer_id"],
    "timestamp_utc": orders_id_unique["timestamp_utc"],
    "type": "Order",
    "CA": orders_id_unique["CA"],
})

# Pubs : rattachées à un client via mapping
cle_dsp = mapping_id.dropna(subset=["dsp_id"]).set_index("dsp_id")["customer_id"]
cle_tv = mapping_id.dropna(subset=["device_id"]).set_index("device_id")["customer_id"]

pub_prog = pd.DataFrame({
    "customer_id": prog_id["dsp_id"].map(cle_dsp),
    "timestamp_utc": prog_id["timestamp_utc"],
    "type": "Pub " + prog_id["campaign_name"].astype(str),
})
pub_tv = pd.DataFrame({
    "customer_id": tv_id["device_id"].map(cle_tv),
    "timestamp_utc": tv_id["timestamp_utc"],
    "type": "Pub TV",
})
print(f"Impressions programmatiques sans client : {pub_prog['customer_id'].isna().sum():,}")
print(f"Impressions TV sans client              : {pub_tv['customer_id'].isna().sum():,}")
# Suppression des impressions sans client identifié
pub_prog = pub_prog.dropna(subset=["customer_id"])
pub_tv = pub_tv.dropna(subset=["customer_id"])

# Assemblage et tri chronologique
parcours = pd.concat([site, commandes, pub_prog, pub_tv], ignore_index=True)
parcours["type"] = pd.Categorical(parcours["type"], categories=ordre_types, ordered=True)
parcours["est_pub"] = parcours["type"].isin(ordre_types[3:])
parcours = parcours.sort_values(["customer_id", "timestamp_utc", "type"], ignore_index=True)
del vp, site, commandes, pub_prog, pub_tv

Vérifications

In [ ]:
# Rien n'a été perdu en route
attendu = pd.Series({
    "Product Page View": (retailer_id["event_name"] == "Product Page View").sum(),
    "Add to cart": (retailer_id["event_name"] == "Add to cart").sum(),
    "Order": len(orders_id_unique),
})
compte = parcours["type"].value_counts().reindex(ordre_types)
print(compte)
assert (compte[attendu.index] == attendu).all()
assert parcours["customer_id"].notna().all() and parcours["type"].notna().all()
assert np.isclose(parcours["CA"].sum(), orders_id_unique["CA"].sum())

print(f"\n{len(parcours):,} lignes, {parcours['customer_id'].nunique():,} personnes")
print(parcours.groupby("type", observed=True)["timestamp_utc"].agg(["min", "max"]))

# Exemple : la frise d'un acheteur exposé à la TV
exemple = parcours.loc[parcours["type"] == "Pub TV", "customer_id"]
exemple = exemple[exemple.isin(orders_id_unique["customer_id"])].iloc[0]
parcours[parcours["customer_id"] == exemple]

- Pubs sans client : 310 k impressions programmatiques et 113 k impressions TV (2 % chacune) ont un identifiant absent de mapping. Si on ajoute les impressions sans identifiant, 87 % des impressions programmatiques et 95 % des impressions TV sont reliées à un client.
- Personnes sans aucun événement : parcours compte 7,28 M personnes sur les 7,83 M de mapping. Environ 0,54 M n’ont ni pub ni visite.
- Le retargeting démarre le 1er janvier à 00:00:04, avant toute visite qu’on puisse observer. Il se base donc sur des visites antérieures à la période. C’est une limite à garder en tête : « première visite » veut dire première visite observée.

`parcours` compte 30,5 M de lignes pour 7,28 M de personnes : environ 0,54 M des personnes de `mapping` n'ont ni pub ni visite. 310 k impressions programmatiques et 113 k impressions TV (2 % de chaque canal) ont un identifiant absent de `mapping` et sont retirées. Au total, 87 % des impressions programmatiques et 95 % des impressions TV sont rattachées à un client.

Le retargeting commence le 1er janvier à 00:00:04, avant toute visite observable : il repose sur des visites antérieures à la période. Dans la suite, « première visite » signifie première visite observée.

#### Table `pop` : une ligne par personne

Une ligne pour chacune des 7,8 M personnes de `mapping`, qu'elles aient acheté ou non. C'est la table qui sert à mesurer l'effet de la publicité.
- exposition par canal : nombre de pubs et date de la première ; `NaN` si la personne n'est pas mesurable sur ce canal, 0 si elle est mesurable mais non exposée ;
- site : nombre de vues, de paniers et de commandes, CA, date de la première visite ;
- avant / après le 29 février (début de la TV) : commandes et CA sur chaque période ;
- `pub_avant_visite_*` : la première pub du canal précède-t-elle la première visite observée ? (`NaN` si la personne n'est pas visiteuse ou pas exposée) ;
- profil (âge, revenu, race) : connu pour les visiteurs uniquement.

- pour début tv il faudrait pas direct prendre le min (2024-02-29 21:00:57) ?
- c'est pas bizarre de comparer l'activité commerciale avant et après le début de la publicité TV sachant qu'on ne sait même pas si la personne a été exposée à la télé ?
- je comprends pas ça "pop["CA"] = pop["CA_avant"] + pop["CA_apres"]" j'ai pas déjà récupéré le CA dans commandes donc periodes ?

In [ ]:
noms = {"Product Page View": "vue", "Add to cart": "panier", "Order": "commande",
        "Pub Contextual": "contextual", "Pub Retargeting": "retargeting", "Pub TV": "tv"}
debut_tv = pd.Timestamp("2024-02-29")   # la TV démarre le 29/02 à 21 h : toute la journée compte « après »
canaux = {"contextual": "dsp_mesurable", "retargeting": "dsp_mesurable", "tv": "tv_mesurable"}

# Nombre d'événements et première date, par personne et par type
compte = (parcours.groupby(["customer_id", "type"], observed=True)["timestamp_utc"]
                  .agg(nb="size", premiere="min").unstack())
compte.columns = [f"{stat}_{noms[t]}" for stat, t in compte.columns]

# Commandes et CA avant / après le début de la TV
commandes = parcours.loc[parcours["type"] == "Order", ["customer_id", "timestamp_utc", "CA"]]
commandes["periode"] = np.where(commandes["timestamp_utc"] < debut_tv, "avant", "apres")
periodes = (commandes.groupby(["customer_id", "periode"])["CA"]
                     .agg(nb_commandes="size", CA="sum").unstack(fill_value=0))
periodes.columns = [f"{stat}_{p}" for stat, p in periodes.columns]

# Une ligne par personne de mapping
pop = (mapping_id[["customer_id", "dsp_mesurable", "tv_mesurable"]]
       .merge(compte, left_on="customer_id", right_index=True, how="left")
       .merge(periodes, left_on="customer_id", right_index=True, how="left")
       .merge(socio_id, on="customer_id", how="left", validate="one_to_one"))
assert len(pop) == len(mapping_id) and pop["customer_id"].is_unique

# Site : 0 si aucun événement
cols_site = ["nb_vue", "nb_panier", "nb_commande", "nb_commandes_avant", "nb_commandes_apres", "CA_avant", "CA_apres"]
pop[cols_site] = pop[cols_site].fillna(0)
pop["CA"] = pop["CA_avant"] + pop["CA_apres"]
pop["premiere_visite"] = pop[["premiere_vue", "premiere_panier", "premiere_commande"]].min(axis=1)
pop["visiteur"] = pop["premiere_visite"].notna()
pop["acheteur"] = pop["nb_commande"] > 0

# Pubs : NaN si non mesurable, 0 si mesurable non exposé
for c, mesurable in canaux.items():
    pop[f"nb_{c}"] = pop[f"nb_{c}"].fillna(0).where(pop[mesurable])
    pop[f"expose_{c}"] = (pop[f"nb_{c}"] > 0).where(pop[mesurable]).astype("boolean")
    pop[f"pub_avant_visite_{c}"] = ((pop[f"premiere_{c}"] < pop["premiere_visite"])
                                    .where(pop["visiteur"] & pop[f"premiere_{c}"].notna())
                                    .astype("boolean"))

Vérifications

In [ ]:
# Rien n'a été perdu entre parcours et pop
nb_par_type = parcours["type"].value_counts()
assert pop["nb_commande"].sum() == nb_par_type["Order"]
assert np.isclose(pop["CA"].sum(), parcours["CA"].sum())
for c, t in [("contextual", "Pub Contextual"), ("retargeting", "Pub Retargeting"), ("tv", "Pub TV")]:
    assert pop[f"nb_{c}"].sum() == nb_par_type[t]
assert pop["visiteur"].sum() == retailer_id["customer_id"].nunique()   # tous les clients du site sont dans mapping
assert (pop["nb_commandes_avant"] + pop["nb_commandes_apres"] == pop["nb_commande"]).all()

# Exposition par canal
resume = pd.DataFrame({
    "Mesurables": [pop[f"expose_{c}"].notna().sum() for c in canaux],
    "Exposés": [pop[f"expose_{c}"].sum() for c in canaux],
    "Exposés (% des mesurables)": [pop[f"expose_{c}"].mean() * 100 for c in canaux],
}, index=list(canaux))
print(resume.round(1))
print(f"\nPersonnes : {len(pop):,} | visiteurs : {pop['visiteur'].sum():,} ({pop['visiteur'].mean():.1%})"
      f" | acheteurs : {pop['acheteur'].sum():,} ({pop['acheteur'].mean():.1%})")

# La question laissée en suspens : qui est retargeté ?
r = pop[pop["expose_retargeting"] == True]
print(f"\nRetargetés : {len(r):,}")
print(f"  sans aucune visite sur la période       : {(~r['visiteur']).mean():.1%}")
print(f"  1re pub retargeting avant la 1re visite : {r['pub_avant_visite_retargeting'].mean():.1%} (parmi les visiteurs)")

**Exposition.** Parmi les personnes mesurables, 48,9 % ont vu au moins une pub contextuelle, 25,6 % une pub retargeting et 26,7 % une pub TV. 17,3 % des personnes de `mapping` sont venues sur le site, 6,4 % ont acheté.

**Qui est retargeté ?** 1,78 M de personnes ont reçu du retargeting, mais 74,0 % d'entre elles ne sont jamais venues sur le site pendant la période. Le ciblage repose donc sur autre chose que les visites observées : visites antérieures au 1er janvier ou audience plus large (hypothèses, non vérifiables avec ces données). Les retargetés venus sur le site (environ 464 k, un tiers des visiteurs) ont reçu, pour 73,6 % d'entre eux, leur première pub retargeting après leur première visite : la pub suit l'intérêt au lieu de le précéder. Comparer retargetés et non-retargetés mélangera donc l'effet de la pub et un intérêt déjà présent.

_____________
|> je sais pas si c'est à mettre

#### Première comparaison : exposés et non-exposés

Chaque personne mesurable sur les deux canaux est classée selon la combinaison de pubs reçues sur la période. Le groupe « Aucune pub » sert de référence. Cette comparaison est volontairement naïve : elle ne tient compte ni de l'ordre dans le temps (un achat peut précéder la pub), ni du ciblage, ni du profil des personnes. Elle sert de point de départ.

In [ ]:
# Profil d'exposition, sur les personnes mesurables sur les deux canaux
m = pop[pop["dsp_mesurable"] & pop["tv_mesurable"]].copy()
m["profil_expo"] = ""
for c, nom in [("contextual", "Contextual"), ("retargeting", "Retargeting"), ("tv", "TV")]:
    m["profil_expo"] += np.where(m[f"expose_{c}"], f" + {nom}", "")
m["profil_expo"] = m["profil_expo"].str.removeprefix(" + ").replace("", "Aucune pub")

g = m.groupby("profil_expo")
comparaison = pd.DataFrame({
    "Personnes": g.size(),
    "Part (%)": g.size() / len(m) * 100,
    "Visiteurs (%)": g["visiteur"].mean() * 100,
    "Acheteurs (%)": g["acheteur"].mean() * 100,
    "CA par personne": g["CA"].mean(),
    "CA par acheteur": g["CA"].sum() / g["acheteur"].sum(),
}).sort_values("Personnes", ascending=False)
ref = comparaison.loc["Aucune pub", "Acheteurs (%)"]
comparaison["Acheteurs vs aucune pub (x)"] = comparaison["Acheteurs (%)"] / ref
comparaison.round(2)

**Lecture.** Tous les groupes exposés comptent beaucoup moins d'acheteurs que le groupe « Aucune pub » (0,85 % à 13,4 % contre 41,7 %). Ce n'est pas un effet négatif de la publicité mais un effet de construction de la base : le groupe « Aucune pub » est composé à 100 % de visiteurs. `mapping` ne contient que des personnes exposées à une pub ou venues sur le site : les non-exposés jamais venus n'y figurent pas, alors que les exposés jamais venus sont nombreux.

**Conséquence.** On ne peut pas comparer exposés et non-exposés sur toute la population. La comparaison se fait entre visiteurs, seule population où les deux groupes sont complets. Elle mesure l'effet de la pub sur l'achat parmi ceux qui viennent sur le site, pas sur le fait de venir.

**Ce qui reste lisible.** À combinaison égale par ailleurs, ajouter le retargeting augmente nettement la part de visiteurs (7,6 % pour Contextual seul, 34,3 % pour Contextual + Retargeting) et d'acheteurs. Les acheteurs retargetés dépensent aussi davantage (116 à 138 $ contre 96 à 112 $). C'est cohérent avec un ciblage de personnes déjà intéressées.

Désormais le seul groupe pour lequel on voit à la fois des exposés et des non-exposés est le groupe des visiteurs donc toutes les comparaisons se feront désormais entre visiteurs.

|<
________________

### 5.4 Comparaison entre visiteurs

`mapping` ne contient que des personnes exposées ou venues sur le site : les non-exposés jamais venus n'y figurent pas. La comparaison se fait donc entre visiteurs, mesurables sur les deux canaux, selon la combinaison de pubs reçues. Elle porte sur l'achat parmi ceux qui viennent sur le site, pas sur le fait de venir.

In [ ]:
# Profil d'exposition (personnes mesurables sur les deux canaux)
m = pop[pop["dsp_mesurable"] & pop["tv_mesurable"]].copy()
m["profil_expo"] = ""
for c, nom in [("contextual", "Contextual"), ("retargeting", "Retargeting"), ("tv", "TV")]:
    m["profil_expo"] += np.where(m[f"expose_{c}"], f" + {nom}", "")
m["profil_expo"] = m["profil_expo"].str.removeprefix(" + ").replace("", "Aucune pub")

# Vérification 1 : personne n'est à la fois non exposé et non visiteur
n_aucun_non_vis = ((m["profil_expo"] == "Aucune pub") & ~m["visiteur"]).sum()
print(f"Non exposés et non visiteurs : {n_aucun_non_vis:,}")

# Vérification 2 : les personnes sans pub mesurée ni visite sont non mesurables sur au moins un canal
sans_rien = pop[~pop["visiteur"] & pop[["nb_contextual", "nb_retargeting", "nb_tv"]].fillna(0).sum(axis=1).eq(0)]
print(f"Personnes sans pub mesurée ni visite : {len(sans_rien):,}")
print(sans_rien[["dsp_mesurable", "tv_mesurable"]].value_counts())

# Tableau par combinaison de pubs
def tableau_expo(df):
    g = df.groupby("profil_expo")
    t = pd.DataFrame({
        "Personnes": g.size(),
        "Part (%)": g.size() / len(df) * 100,
        "Acheteurs (%)": g["acheteur"].mean() * 100,
        "CA par personne": g["CA"].mean(),
        "CA par acheteur": g["CA"].sum() / g["acheteur"].sum(),
        "Commandes par acheteur": g["nb_commande"].sum() / g["acheteur"].sum(),
    }).sort_values("Personnes", ascending=False)
    t["Acheteurs vs aucune pub (x)"] = t["Acheteurs (%)"] / t.loc["Aucune pub", "Acheteurs (%)"]
    return t

visiteurs = m[m["visiteur"]]
print(f"\nVisiteurs mesurables sur les deux canaux : {len(visiteurs):,}")
tableau_expo(visiteurs).round(2)

**Vérifications.** Seules 10 personnes mesurables sont à la fois non exposées et non visiteuses. Les 540 934 personnes sans pub mesurée ni visite sont, à ces 10 exceptions près, non mesurables sur au moins un canal : elles figurent dans `mapping` parce qu'elles ont été exposées sur le canal qu'on ne voit pas pour elles. La construction de la base est confirmée.

**Lecture du tableau (1,17 M de visiteurs).** 35 % des visiteurs n'ont reçu aucune pub mesurée ; ils servent de référence (41,7 % d'acheteurs, 46,75 $ de CA par personne).
- **Retargeting** : les visiteurs retargetés achètent plus souvent (48 % d'acheteurs), commandent plus (3,1 commandes par acheteur contre 2,6) et génèrent plus de CA (66 $ par personne contre 47 $). Mais 3 sur 4 ont reçu leur première pub retargeting après leur première visite : ce sont des visiteurs déjà actifs, et l'écart mesure surtout leur intérêt.
- **Contextuel** : les visiteurs exposés achètent beaucoup moins (11 % d'acheteurs), et ajouter le contextuel fait baisser la part d'acheteurs dans toutes les combinaisons. Hypothèse : la pub contextuelle amène sur le site des personnes peu décidées à acheter, ce qui fait baisser le taux d'achat des visiteurs exposés. À vérifier : la pub contextuelle précède-t-elle la première visite ?
- **TV** : écart quasi nul (45 % d'acheteurs contre 42 %, 47 $ de CA par personne dans les deux cas). La TV n'est diffusée qu'à partir du 29 février alors que les achats couvrent tout le semestre : la comparaison avant / après est plus adaptée pour la juger.

Aucun de ces écarts ne peut être lu comme un effet de la publicité : les groupes diffèrent par le ciblage, par l'ordre dans le temps et peut-être par le profil.

### 5.5 Tenir compte du profil

Les groupes d'exposition n'ont pas forcément la même composition en âge et en revenu, deux variables qui influencent l'achat (partie 4). On compare donc chaque groupe aux visiteurs sans pub par standardisation indirecte : nombre d'acheteurs observés / nombre d'acheteurs attendus si le groupe achetait comme les visiteurs sans pub de même âge et de même revenu (30 cases âge × revenu). Même calcul pour le CA. Si le ratio standardisé est proche du ratio brut, le profil n'explique pas l'écart.

In [ ]:
assert visiteurs[["age", "income"]].notna().all().all()   # profil connu pour tous les visiteurs

# 1. Composition des groupes (% de chaque groupe)
compo_age = pd.crosstab(visiteurs["profil_expo"], visiteurs["age"], normalize="index") * 100
compo_revenu = pd.crosstab(visiteurs["profil_expo"], visiteurs["income"], normalize="index") * 100
display(compo_age.round(1))
display(compo_revenu.round(1))

# 2. Taux d'achat et CA moyen de la référence (visiteurs sans pub), par case âge x revenu
ref = visiteurs[visiteurs["profil_expo"] == "Aucune pub"]
taux_ref = (ref.groupby(["age", "income"], observed=True)
               .agg(p_achat_ref=("acheteur", "mean"), ca_moyen_ref=("CA", "mean")).reset_index())

# 3. Acheteurs et CA attendus dans chaque groupe, case par case
cases = (visiteurs.groupby(["profil_expo", "age", "income"], observed=True)
                  .agg(n=("acheteur", "size"), acheteurs=("acheteur", "sum"), CA=("CA", "sum"))
                  .reset_index()
                  .merge(taux_ref, on=["age", "income"], how="left"))
assert cases["p_achat_ref"].notna().all()
cases["acheteurs_attendus"] = cases["n"] * cases["p_achat_ref"]
cases["CA_attendu"] = cases["n"] * cases["ca_moyen_ref"]

# 4. Ratios bruts et standardisés
t = cases.groupby("profil_expo")[["n", "acheteurs", "acheteurs_attendus", "CA", "CA_attendu"]].sum()
standard = pd.DataFrame({
    "Personnes": t["n"],
    "Acheteurs (%)": t["acheteurs"] / t["n"] * 100,
    "Ratio brut acheteurs (x)": t["acheteurs"] / t["n"] / ref["acheteur"].mean(),
    "Ratio standardisé acheteurs (x)": t["acheteurs"] / t["acheteurs_attendus"],
    "CA par personne": t["CA"] / t["n"],
    "Ratio brut CA (x)": t["CA"] / t["n"] / ref["CA"].mean(),
    "Ratio standardisé CA (x)": t["CA"] / t["CA_attendu"],
}).sort_values("Personnes", ascending=False)
assert np.isclose(standard.loc["Aucune pub", "Ratio standardisé acheteurs (x)"], 1)
standard.round(2)

**Composition.** Les groupes ont presque la même répartition par âge et par revenu (2 points d'écart au plus). Les groupes exposés au contextuel comptent un peu plus de 65 ans et un peu moins de 35-45 ans, tranches qui achètent respectivement le moins et le plus (partie 4), mais l'écart reste faible.

**Ratios standardisés.** Ils sont identiques aux ratios bruts à 0,01 près, pour la part d'acheteurs comme pour le CA. L'âge et le revenu n'expliquent donc aucune part des écarts entre groupes : l'exposition à la publicité ne dépend pas du profil socio-démographique. Les écarts viennent d'autre chose : le ciblage sur le comportement (retargeting), l'ordre dans le temps, ou le type de visiteurs que chaque canal amène sur le site.

### 5.6 L'ordre dans le temps

Pour chaque canal, on regarde si la pub arrive avant ou après la première visite observée, puis on compare l'achat des visiteurs exposés avant et après leur première visite. Pour ne pas mélanger des groupes exposés à plusieurs canaux, cette comparaison porte sur les visiteurs exposés à un seul canal. Un visiteur arrivé tard dans la période a moins de temps pour acheter : les ratios sont donc standardisés sur le mois de la première visite (référence : visiteurs sans pub).

In [ ]:
canaux_noms = {"contextual": "Contextual", "retargeting": "Retargeting", "tv": "TV"}

# 1. Ordre dans le temps, par canal (tous les visiteurs exposés au canal)
ordre = {}
for c, nom in canaux_noms.items():
    expo = visiteurs[visiteurs[f"expose_{c}"] == True]
    ach = expo[expo["acheteur"]]
    ordre[nom] = {
        "Visiteurs exposés": len(expo),
        "Pub avant la 1re visite (%)": expo[f"pub_avant_visite_{c}"].mean() * 100,
        "Acheteurs exposés": len(ach),
        "Pub avant le 1er achat (%)": (ach[f"premiere_{c}"] < ach["premiere_commande"]).mean() * 100,
    }
display(pd.DataFrame(ordre).round(1))

# 2. Achat selon que la pub arrive avant ou après la 1re visite
visiteurs = visiteurs.copy()
visiteurs["mois_1re_visite"] = visiteurs["premiere_visite"].dt.to_period("M")
ref = visiteurs[visiteurs["profil_expo"] == "Aucune pub"]
ref_mois = ref.groupby("mois_1re_visite").agg(p_achat_ref=("acheteur", "mean"), ca_moyen_ref=("CA", "mean"))

def ligne(df):
    """Taux d'achat et CA d'un groupe, bruts et standardisés sur le mois de 1re visite."""
    par_mois = df.groupby("mois_1re_visite").agg(n=("acheteur", "size")).join(ref_mois, how="left")
    assert par_mois["p_achat_ref"].notna().all()
    return pd.Series({
        "Personnes": len(df),
        "Acheteurs (%)": df["acheteur"].mean() * 100,
        "Ratio brut acheteurs (x)": df["acheteur"].mean() / ref["acheteur"].mean(),
        "Ratio standardisé acheteurs (x)": df["acheteur"].sum() / (par_mois["n"] * par_mois["p_achat_ref"]).sum(),
        "CA par personne": df["CA"].mean(),
        "Ratio brut CA (x)": df["CA"].mean() / ref["CA"].mean(),
        "Ratio standardisé CA (x)": df["CA"].sum() / (par_mois["n"] * par_mois["ca_moyen_ref"]).sum(),
    })

groupes = {"Aucune pub": ref}
for c, nom in canaux_noms.items():
    seul = visiteurs[visiteurs["profil_expo"] == nom]
    groupes[f"{nom} seul, pub avant 1re visite"] = seul[seul[f"pub_avant_visite_{c}"] == True]
    groupes[f"{nom} seul, pub après 1re visite"] = seul[seul[f"pub_avant_visite_{c}"] == False]

avant_apres = pd.DataFrame({nom: ligne(df) for nom, df in groupes.items()}).T
avant_apres.round(2)

**Ordre dans le temps.** Parmi les acheteurs exposés, la pub a précédé le premier achat pour 58 % des exposés au contextuel, 42 % des retargetés et 26 % des exposés à la TV (qui ne démarre que le 29 février). Pour les autres, le premier achat a eu lieu avant toute pub de ce canal.

**Contextuel.** Les visiteurs exposés au contextuel achètent aussi peu, que la pub ait précédé leur première visite (ratio standardisé de 0,26) ou l'ait suivie (0,28). Le faible taux d'achat ne vient donc pas de visiteurs peu décidés que la pub aurait amenés : il caractérise le public touché par le contextuel. Rien n'indique que le contextuel augmente les achats des visiteurs.

**Retargeting.** Les visiteurs retargetés avant leur première visite achètent comme les visiteurs sans pub (0,98). L'écart positif vient uniquement de ceux retargetés après leur visite (1,10 en acheteurs, 1,25 en CA). Le retargeting ciblant les visiteurs actifs, cet écart mélange un éventuel effet de rappel et l'intérêt déjà présent ; ces données ne permettent pas de les séparer.

**TV.** Sans correction, les visiteurs exposés à la TV avant leur première visite semblent acheter moins (0,84) : venus après le 29 février, ils ont eu moins de temps pour acheter. Une fois standardisés sur le mois de la première visite, les deux groupes TV achètent environ 10 % de plus que les visiteurs sans pub (1,09 à 1,13) et génèrent 6 à 7 % de CA en plus. Cet écart modeste, mais cohérent, est testé à l'étape suivante avec la comparaison avant / après.

### 5.7 Effet de la TV : comparaison avant / après le 29 février

La TV démarre le 29 février : avant cette date, aucun visiteur n'y a été exposé. On compare l'évolution du CA par personne et par jour entre janvier-février (59 jours) et mars-juin (123 jours), chez les visiteurs exposés à la TV et chez les non-exposés (différence de différences). Cette méthode suppose que les deux groupes auraient évolué de la même façon sans la TV : on le vérifie sur la courbe hebdomadaire avant le 29 février. L'analyse est refaite sur les seuls visiteurs déjà venus avant le 29 février, pour écarter l'effet des nouveaux visiteurs. On vérifie enfin que la pression programmatique a évolué de la même façon dans les deux groupes.

In [ ]:
debut_tv = pd.Timestamp("2024-02-29")
jours_avant, jours_apres = 59, 123   # 1er janvier - 28 février / 29 février - 30 juin

def did_tv(df, titre):
    """Différence de différences sur le CA par personne et par jour."""
    d = df.assign(
        groupe=np.where(df["expose_tv"] == True, "Exposés TV", "Non exposés"),
        acheteur_avant=df["nb_commandes_avant"] > 0,
        acheteur_apres=df["nb_commandes_apres"] > 0,
        ca_jour_avant=df["CA_avant"] / jours_avant,
        ca_jour_apres=df["CA_apres"] / jours_apres,
    )
    d["evolution"] = d["ca_jour_apres"] - d["ca_jour_avant"]
    g = d.groupby("groupe")
    t = pd.DataFrame({
        "Personnes": g.size(),
        "Acheteurs avant (%)": g["acheteur_avant"].mean() * 100,
        "Acheteurs après (%)": g["acheteur_apres"].mean() * 100,
        "CA / pers. / jour avant": g["ca_jour_avant"].mean(),
        "CA / pers. / jour après": g["ca_jour_apres"].mean(),
        "Évolution": g["evolution"].mean(),
    })
    e = d.loc[d["groupe"] == "Exposés TV", "evolution"]
    n = d.loc[d["groupe"] == "Non exposés", "evolution"]
    did = e.mean() - n.mean()
    se = np.sqrt(e.var() / len(e) + n.var() / len(n))
    sans_tv = t.loc["Exposés TV", "CA / pers. / jour avant"] + t.loc["Non exposés", "Évolution"]
    print(f"\n{titre}")
    display(t.round(3))
    print(f"Effet TV : {did:+.3f} $ par personne et par jour "
          f"[IC 95 % : {did - 1.96 * se:+.3f} ; {did + 1.96 * se:+.3f}], "
          f"soit {did / sans_tv:+.1%} du CA attendu sans TV")
    print(f"CA supplémentaire sur mars-juin : {did * jours_apres:+.2f} $ par exposé, "
          f"{did * jours_apres * len(e):+,.0f} $ pour les {len(e):,} visiteurs exposés")
    return did * jours_apres * len(e)

ca_sup = did_tv(visiteurs, "Tous les visiteurs")
did_tv(visiteurs[visiteurs["premiere_visite"] < debut_tv], "Visiteurs déjà venus avant le 29 février")

budget_tv = tv_c["cost"].sum()
print(f"\nBudget TV : {budget_tv:,.0f} $ | CA supplémentaire / budget : {ca_sup / budget_tv:.2f} $ par $ dépensé")

In [ ]:
groupe = pd.Series(np.where(visiteurs["expose_tv"] == True, "Exposés TV", "Non exposés"),
                   index=visiteurs["customer_id"])
taille = groupe.value_counts()

# CA par personne et par semaine, dans chaque groupe
cmd_sem = parcours.loc[parcours["type"] == "Order", ["customer_id", "timestamp_utc", "CA"]]
cmd_sem["groupe"] = cmd_sem["customer_id"].map(groupe)
cmd_sem = cmd_sem.dropna(subset=["groupe"])
cmd_sem["semaine"] = cmd_sem["timestamp_utc"].dt.to_period("W").dt.start_time
hebdo = cmd_sem.groupby(["semaine", "groupe"])["CA"].sum().unstack() / taille

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
hebdo.plot(ax=axes[0], marker="o")
axes[0].set_title("CA par personne et par semaine"); axes[0].set_ylabel("$")
(hebdo["Exposés TV"] - hebdo["Non exposés"]).plot(ax=axes[1], marker="o", color="black")
axes[1].axhline(0, color="grey", lw=0.8)
axes[1].set_title("Écart exposés TV - non exposés"); axes[1].set_ylabel("$")
for ax in axes:
    ax.axvline(debut_tv, color="red", ls="--", label="Début TV")
    ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

# Pression programmatique avant / après, dans chaque groupe
pubs_prog = parcours.loc[parcours["type"].isin(["Pub Contextual", "Pub Retargeting"]), ["customer_id", "timestamp_utc"]]
pubs_prog["groupe"] = pubs_prog["customer_id"].map(groupe)
pubs_prog = pubs_prog.dropna(subset=["groupe"])
pubs_prog["periode"] = np.where(pubs_prog["timestamp_utc"] < debut_tv, "avant", "apres")
pression = pubs_prog.groupby(["groupe", "periode"]).size().unstack().div(taille, axis=0)
pression["avant"] /= jours_avant
pression["apres"] /= jours_apres
print("Impressions programmatiques par personne et par jour :")
display(pression[["avant", "apres"]].round(4))

**Tous les visiteurs.** Avant le 29 février, les deux groupes ont le même CA par personne et par jour (0,255 $ et 0,259 $) et l'écart hebdomadaire oscille autour de 0 : l'hypothèse d'évolution parallèle est plausible. Après le 29 février, l'écart devient positif et le reste jusqu'en juin (autour de 0,2 $ par personne et par semaine). La différence de différences est de +0,033 $ par personne et par jour (IC 95 % : +0,029 ; +0,037), soit +14 % du CA attendu sans TV, environ 414 k$ de CA supplémentaire ou 3,1 $ de CA par dollar de budget TV.

**Visiteurs déjà venus avant le 29 février.** Le résultat s'inverse : −0,017 $ par personne et par jour (IC 95 % : −0,024 ; −0,010), soit −4,6 %. Les deux groupes baissent fortement après le 29 février, ce qui est attendu : ils ont été retenus parce qu'ils étaient actifs avant. Les exposés partaient d'un niveau plus élevé (0,550 $ contre 0,481 $) et, en proportion, leur baisse est même un peu plus faible (−35 % contre −36 %). Chez les clients déjà présents, l'effet de la TV est nul ou très faible.

**Pourquoi les deux résultats divergent.** L'effet positif de la première analyse vient des visiteurs arrivés après le 29 février, plus nombreux chez les exposés (54 % contre 46 %). Il peut refléter des visiteurs amenés par la TV, mais pas seulement : pour environ 47 % de ces nouveaux visiteurs exposés, la première pub TV est arrivée après leur première visite. Les 414 k$ ne peuvent donc pas être attribués à la TV sans vérification supplémentaire.

**Programmatique.** La pression programmatique baisse de moitié dans les deux groupes après le 29 février (−48 % et −54 %) : elle évolue de façon comparable et n'explique pas l'écart.

### 5.8 La TV amène-t-elle de nouveaux clients ?

Les visiteurs ont été choisis parce qu'ils sont venus sur le site : cette population ne permet pas de savoir si la TV fait venir de nouvelles personnes. On reprend la comparaison avant / après sur les personnes exposées au contextuel, présentes dans `mapping` qu'elles soient venues ou non sur le site (les non-visiteurs comptent pour 0). On compare le CA par personne, puis le taux de premières visites et de premiers achats parmi les personnes qui ne sont pas encore venues ou n'ont pas encore acheté.

In [ ]:
ctx = m[m["expose_contextual"] == True].copy()
ctx["groupe"] = np.where(ctx["expose_tv"] == True, "Exposés TV", "Non exposés")
print(ctx["groupe"].value_counts())

# Contrôle : les deux groupes reçoivent-ils le contextuel aux mêmes moments ?
ctx["contextuel_apres"] = ctx["premiere_contextual"] >= debut_tv
print("1re pub contextuelle après le 29 février (%) :")
display((ctx.groupby("groupe")["contextuel_apres"].mean() * 100).round(1))

# 1. CA par personne et par jour (non-visiteurs compris)
did_tv(ctx, "Exposés au contextuel, visiteurs ou non")

# 2. Nouveaux visiteurs et nouveaux acheteurs
def nouveaux(df, col, nom):
    """Taux de 1re visite (ou 1er achat) par jour, parmi ceux qui ne l'ont pas encore fait."""
    lignes = {}
    for g, d in df.groupby("groupe"):
        k_avant = (d[col] < debut_tv).sum()
        k_apres = (d[col] >= debut_tv).sum()
        lignes[g] = {
            "k_avant": k_avant, "k_apres": k_apres,
            "Taux avant (pour 1 000 / jour)": k_avant / (len(d) * jours_avant) * 1000,
            "Taux après (pour 1 000 / jour)": k_apres / ((len(d) - k_avant) * jours_apres) * 1000,
        }
    t = pd.DataFrame(lignes).T
    t["Évolution (x)"] = t["Taux après (pour 1 000 / jour)"] / t["Taux avant (pour 1 000 / jour)"]
    rapport = t.loc["Exposés TV", "Évolution (x)"] / t.loc["Non exposés", "Évolution (x)"]
    se = np.sqrt((1 / t[["k_avant", "k_apres"]].astype(float)).to_numpy().sum())
    print(f"\n{nom} : évolution des exposés TV / évolution des non exposés = {rapport:.2f} "
          f"[IC 95 % : {rapport * np.exp(-1.96 * se):.2f} ; {rapport * np.exp(1.96 * se):.2f}]")
    display(t.drop(columns=["k_avant", "k_apres"]).round(3))

nouveaux(ctx, "premiere_visite", "Nouveaux visiteurs")
nouveaux(ctx, "premiere_commande", "Nouveaux acheteurs")

In [ ]:
groupe_ctx = pd.Series(ctx["groupe"].to_numpy(), index=ctx["customer_id"])
taille_ctx = groupe_ctx.value_counts()

# CA par personne et par semaine
cmd_sem = parcours.loc[parcours["type"] == "Order", ["customer_id", "timestamp_utc", "CA"]]
cmd_sem["groupe"] = cmd_sem["customer_id"].map(groupe_ctx)
cmd_sem = cmd_sem.dropna(subset=["groupe"])
cmd_sem["semaine"] = cmd_sem["timestamp_utc"].dt.to_period("W").dt.start_time
ca_sem = cmd_sem.groupby(["semaine", "groupe"])["CA"].sum().unstack() / taille_ctx

# Nouveaux visiteurs par semaine, pour 1 000 personnes
nv = ctx.dropna(subset=["premiere_visite"])
nv_sem = (nv.groupby([nv["premiere_visite"].dt.to_period("W").dt.start_time, "groupe"]).size()
            .unstack() / taille_ctx * 1000)

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
(ca_sem["Exposés TV"] - ca_sem["Non exposés"]).plot(ax=axes[0], marker="o", color="black")
axes[0].axhline(0, color="grey", lw=0.8)
axes[0].set_title("Écart de CA par personne et par semaine (exposés TV - non exposés)"); axes[0].set_ylabel("$")
nv_sem.plot(ax=axes[1], marker="o")
axes[1].set_title("Nouveaux visiteurs pour 1 000 personnes, par semaine")
for ax in axes:
    ax.axvline(debut_tv, color="red", ls="--", label="Début TV")
    ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

**Population.** 3,3 M de personnes exposées au contextuel, dont 82 k exposées à la TV. Les deux groupes reçoivent le contextuel à des moments assez proches (première pub contextuelle après le 29 février : 37 % contre 42 %). Avant la TV, les exposés TV achetaient déjà 2,5 fois plus souvent (1,5 % d'acheteurs contre 0,6 %) : la comparaison porte sur l'évolution, pas sur le niveau.

**CA.** L'effet mesuré est faible : +0,001 $ par personne et par jour (IC 95 % : 0,000 ; 0,002), soit +7,5 % du CA attendu sans TV. Sur la courbe hebdomadaire, l'écart entre les deux groupes reste dans la même fourchette avant et après le 29 février (0,055 à 0,09 $ par semaine) ; il augmente un peu en moyenne (de 0,067 à 0,075 $), sans rupture visible au démarrage de la TV.

**Nouveaux clients.** Le taux de premières visites baisse un peu moins chez les exposés TV après le 29 février (rapport de 1,06 ; IC 95 % : 1,03 ; 1,10) : la TV pourrait amener un peu plus de nouveaux visiteurs (+6 %). Ce surplus ne se traduit pas en nouveaux acheteurs (rapport de 0,93 ; IC 95 % : 0,86 ; 1,01).

**Bilan sur la TV.** Selon la population étudiée, l'effet mesuré va de −4,6 % (clients déjà présents) à +14 % (tous les visiteurs, effet porté par les nouveaux visiteurs), en passant par +7,5 % sans rupture visible (exposés au contextuel). Le résultat le plus solide est une légère hausse des premières visites, sans hausse des nouveaux acheteurs. Les données ne permettent donc pas de conclure que la TV augmente les ventes : si un effet existe, il est faible, et l'estimation de 3,1 $ de CA par dollar dépensé n'est pas fiable.

### 5.9 Conclusion : la publicité fait-elle vendre ?

**Réponse courte.** Avec ces données, on ne peut pas montrer que la publicité fait vendre : aucun canal ne présente d'effet sur les ventes qui résiste aux vérifications. Cela ne prouve pas qu'elle est inutile : la base ne contient pas de groupe de comparaison qui permettrait de mesurer son effet propre.

| Canal | Budget | Ce qu'on observe | Ce qu'on peut conclure |
|---|---|---|---|
| TV | 135 k$ (59 %) | Effet sur le CA de −4,6 % à +14 % selon la population, sans rupture visible au démarrage de la TV ; +6 % de premières visites, mais pas plus de nouveaux acheteurs | Pas d'effet robuste sur les ventes ; au mieux un léger effet sur la venue sur le site |
| Retargeting | ≈ 42 k$ (18 %) | Les visiteurs retargetés achètent plus (48 % d'acheteurs contre 42 %, 66 $ de CA par personne contre 47 $), mais 3 sur 4 ont été retargetés après leur première visite ; ceux retargetés avant achètent comme les autres, et 58 % des acheteurs retargetés avaient acheté avant leur première pub | L'écart vient du ciblage de visiteurs déjà actifs ; l'effet propre n'est pas mesurable |
| Contextuel | ≈ 51 k$ (22 %) | Les visiteurs exposés achètent près de 4 fois moins (11 % d'acheteurs contre 42 %), que la pub précède ou suive leur première visite | Le public touché achète peu ; aucun signe d'effet positif |

L'âge et le revenu n'expliquent aucun de ces écarts : les groupes ont la même composition et les ratios standardisés sont identiques aux ratios bruts.

**Rentabilité.** Le budget média s'élève à 228 k$, soit 0,37 % du CA de la période. Pour que chaque dollar dépensé rapporte au moins un dollar de CA, la publicité aurait dû générer 228 k$ de ventes supplémentaires. Ce n'est pas démontré. L'estimation de 3,1 $ de CA par dollar de TV repose sur une population où l'effet ne tient pas ; pour le programmatique, aucune estimation fiable n'est possible.

**Limites.**
- `mapping` ne contient pas de personnes non exposées et jamais venues sur le site : les comparaisons se limitent aux visiteurs, ce qui ne mesure pas l'effet de la pub sur la venue.
- Six mois d'observation seulement : le retargeting démarre dès le 1er janvier sur des visites antérieures, et « première visite » signifie première visite observée.
- La mesure est incomplète : 11 % du budget programmatique et 3 % du budget TV portent sur des impressions sans identifiant, et environ 2 % des impressions de chaque canal ne sont reliées à aucun client.
- L'exposition TV est mesurée par box, donc par foyer, et non par personne.
- Les critères de ciblage du contextuel et du retargeting ne sont pas connus ; les résultats sont des associations, pas des causes.
- Les montants sont du CA, pas de la marge.

**Recommandations.**
1. **Mesurer l'effet par un test avec groupe témoin tiré au hasard** : une part de l'audience ciblée ne reçoit pas la pub (ou reçoit une pub neutre), et l'on compare ses ventes à celles des exposés. C'est la seule façon de mesurer l'effet du retargeting, dont le ciblage fausse toute comparaison.
2. **Commencer par le retargeting** : il absorbe 45 % du budget programmatique, coûte environ 3 fois plus cher à l'impression que le contextuel, et son effet n'est pas démontré.
3. **Pour la TV**, appliquer le même principe avec des foyers ou des zones géographiques témoins.
4. **Compléter les données** : inclure des personnes non exposées et jamais venues sur le site, un historique antérieur au début des campagnes, la règle et la date de déclenchement du retargeting, et la marge par produit.
5. **Examiner les contextes de diffusion du contextuel** : le public qu'il touche achète près de 4 fois moins que les autres visiteurs.